# Linux Gaming & Game-Development Environment

Temporary visual testing environment and Linux gaming storefront bench on Kaggle.

### Architecture & Storefront Suite
- **Unprivileged Execution**: Storefronts and games execute under standard user `gamer` for credential protection and clean user space.
- **Steam (Valve Native)**: Full multiarch client running with software CEF flags. License agreement presented directly to the user in the GUI.
- **Epic Games**: Heroic GUI and Legendary CLI are available for testing. Launch each installed title to verify compatibility.
- **Wine Runner**: Heroic settings select a complete Wine runner, disable UMU/DXVK/VKD3D for this container, and back up existing JSON before repair.
- **GOG Galaxy (Minigalaxy & Heroic)**: DRM-free client for GOG games. Supports native Linux builds and Wine.
- **Lutris Gaming Hub**: Multi-storefront aggregator installed from Ubuntu multiverse.
- **EA App (experimental)**: Wine installer opens; login and game execution need interactive validation.
- **Controller-Friendly Launcher**: Pygame launcher supporting userspace Xbox controllers, keyboard, and mouse navigation.
- **Desktop Window Management**: Openbox window manager and tint2 taskbar panel for smooth multi-window multitasking.
- **Selkies 2.1.0 Streaming**: Tesla T4 NVENC hardware H.264 video encoding at 60 FPS, Opus stereo audio, and 4 virtual Xbox gamepad sockets over Kaggle signed Jupyter Server proxy.

### Hardware Reality & Constraints
- **NVENC Hardware Acceleration**: Encoding is hardware-accelerated on Tesla T4 via NVENC.
- **Desktop 3D Rendering**: Rendered via Mesa llvmpipe (software rasterization) because Kaggle datacenter containers lack NVIDIA display and render nodes (`/dev/nvidia-modeset`, `/dev/dri/renderD128`). Native Vulkan and DXVK hardware games are unavailable.

### Instructions
1. Set **Accelerator: GPU T4 x2** and **Internet: On** in the Kaggle sidebar.
2. Run all cells in an active interactive session.
3. Copy the **VSCode Compatible URL** from **Run → Kaggle Jupyter Server**, paste it into the connection cell, and connect!


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

BENCH = Path('/tmp/kaggle-game-dev-bench')
BUILDS = Path('/tmp/game-builds')
LOGS = BENCH / 'logs'
EA_STORE = Path('/tmp/ea-store')
REPO_URL = 'https://github.com/Mohammedsbastawy/kaggle-gaming-bench.git'

BUILDS.mkdir(parents=True, exist_ok=True)
EA_STORE.mkdir(parents=True, exist_ok=True)

# 1. Automatic recovery for apt/dpkg locks (prevents RuntimeError exit 100 on interactive sessions)
print('▶ Clearing stale apt/dpkg locks and recovering package manager...', flush=True)
for lock in ['/var/lib/dpkg/lock-frontend', '/var/lib/dpkg/lock', '/var/lib/apt/lists/lock', '/var/cache/apt/archives/lock']:
    subprocess.run(['fuser', '-k', lock], check=False)
    try:
        os.remove(lock)
    except OSError:
        pass
subprocess.run(['dpkg', '--configure', '-a'], check=False, env=dict(os.environ, DEBIAN_FRONTEND='noninteractive'))

# 2. Clone or update the ready Git repository
setup_sh = BENCH / 'setup.sh'
if not setup_sh.exists():
    shutil.rmtree(str(BENCH), ignore_errors=True)
    print(f'▶ Cloning ready gaming bench suite from {REPO_URL}...', flush=True)
    res = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(BENCH)], timeout=60)
    if res.returncode != 0 or not setup_sh.exists():
        print('Notice: Git clone unavailable, using embedded setup suite.', flush=True)
        BENCH.mkdir(parents=True, exist_ok=True)
        setup_sh.write_text('#!/bin/bash\nset -e\n\nBENCH="/tmp/kaggle-game-dev-bench"\nLOGS="/home/gamer/logs"\nBUILDS="/tmp/game-builds"\nmkdir -p "$BENCH/logs" "$BUILDS" /tmp/ea-store\n\nexport DEBIAN_FRONTEND=noninteractive\nexport PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"\n\necho "=========================================================="\necho "    KAGGLE GAMING & GAME-DEV BENCH BOOTSTRAPPER           "\necho "=========================================================="\n\n# 1. Kill stale apt/dpkg locks if running in dirty interactive session\necho "▶ Checking and clearing stale apt locks..."\nfuser -k /var/lib/dpkg/lock-frontend 2>/dev/null || true\nfuser -k /var/lib/dpkg/lock 2>/dev/null || true\nfuser -k /var/lib/apt/lists/lock 2>/dev/null || true\nfuser -k /var/cache/apt/archives/lock 2>/dev/null || true\nrm -f /var/lib/dpkg/lock* /var/lib/apt/lists/lock* /var/cache/apt/archives/lock* 2>/dev/null || true\ndpkg --configure -a 2>/dev/null || true\n\n# Helper function for resilient apt install with timeout and retries\napt_install() {\n    local label="$1"\n    shift\n    echo "▶ Installing $label..."\n    local attempts=0\n    while [ $attempts -lt 3 ]; do\n        attempts=$((attempts + 1))\n        if apt-get install -y --no-install-recommends \\\n            -o DPkg::Lock::Timeout=120 \\\n            -o APT::Get::Assume-Yes=true \\\n            "$@"; then\n            return 0\n        fi\n        echo "Notice: apt install for $label failed (attempt $attempts/3). Retrying in 4s..."\n        sleep 4\n        fuser -k /var/lib/dpkg/lock-frontend 2>/dev/null || true\n        rm -f /var/lib/dpkg/lock* 2>/dev/null || true\n        dpkg --configure -a 2>/dev/null || true\n    done\n    echo "Warning: Could not install some packages for $label, continuing..."\n    return 0\n}\n\n# 2. Multiarch & repositories setup\ndpkg --add-architecture i386 || true\nif [ -f /etc/apt/sources.list.d/ubuntu.sources ]; then\n    sed -i \'s/Components: main universe/Components: main universe multiverse/g\' /etc/apt/sources.list.d/ubuntu.sources || true\nfi\n\n# Resilient apt update\napt-get update -o DPkg::Lock::Timeout=120 -qq || true\n\n# 3. Core desktop, X11, and audio utilities\napt_install "core desktop" \\\n    xvfb openbox tint2 xterm pulseaudio pulseaudio-utils xauth dbus-x11 \\\n    libgl1-mesa-dri libgbm1 libpixman-1-0 libxkbcommon0 mesa-utils x11-utils \\\n    xdotool wmctrl wget curl cabextract p7zip-full\n\n# Build tools (isolated so multilib issues don\'t block core)\napt_install "build tools" git make gcc\napt-get install -y --no-install-recommends -o DPkg::Lock::Timeout=120 gcc-multilib 2>/dev/null || true\n\n# 4. Wine and complete 32-bit/64-bit graphics runtime\napt_install "wine runtime" wine wine64 winbind\napt_install "32-bit graphics runtime" \\\n    wine32:i386 libgl1:i386 libglx0:i386 libglx-mesa0:i386 libgl1-mesa-dri:i386 \\\n    mesa-vulkan-drivers:i386 libx11-6:i386 libxext6:i386 libxfixes3:i386 \\\n    libxcomposite1:i386 libxrandr2:i386 libxinerama1:i386 libxi6:i386 libasound2:i386 \\\n    libpulse0:i386 libgnutls30:i386 libvulkan1:i386\n\n# 5. Extra libraries for Lutris and Wine Gecko\napt_install "extra libraries" python3-tk python3.12 python3-dbus python3-gi python3-yaml python3-lxml python3-requests python3-pil\napt-get install -y --no-install-recommends -o DPkg::Lock::Timeout=120 wine-gecko 2>/dev/null || true\n\n# 6. Python streaming, interface and gaming packages\npython3 -m pip install --no-cache-dir \\\n    \'selkies[jupyter]==2.1.0\' \'pygame==2.6.1\' \'psutil\' \'legendary-gl\'\n\n# 7. Steam installation (Valve official deb)\nSTEAM_DEB="/tmp/steam_latest.deb"\nif [ ! -f "$STEAM_DEB" ] || [ $(stat -c%s "$STEAM_DEB" 2>/dev/null || echo 0) -lt 1000000 ]; then\n    wget -q https://repo.steampowered.com/steam/archive/stable/steam_latest.deb -O "$STEAM_DEB" || true\nfi\nif [ -f "$STEAM_DEB" ] && [ $(stat -c%s "$STEAM_DEB") -gt 1000000 ]; then\n    apt_install "steam deb" "$STEAM_DEB"\nelse\n    apt_install "steam apt" steam\nfi\n\n# 8. Heroic Games Launcher\nHEROIC_DEB="/tmp/heroic.deb"\nif [ ! -f "$HEROIC_DEB" ] || [ $(stat -c%s "$HEROIC_DEB" 2>/dev/null || echo 0) -lt 10000000 ]; then\n    wget -q https://github.com/Heroic-Games-Launcher/HeroicGamesLauncher/releases/download/v2.22.4/Heroic-2.22.4-linux-amd64.deb -O "$HEROIC_DEB" || true\nfi\nif [ -f "$HEROIC_DEB" ] && [ $(stat -c%s "$HEROIC_DEB") -gt 10000000 ]; then\n    dpkg -i "$HEROIC_DEB" || apt-get install -f -y -o DPkg::Lock::Timeout=120 || true\nfi\n\n# 9. Lutris & Minigalaxy\napt_install "lutris" lutris\napt_install "minigalaxy" minigalaxy\n\n# 10. EA App installer download\nEA_EXE="/tmp/ea-store/EAappInstaller.exe"\nif [ ! -f "$EA_EXE" ] || [ $(stat -c%s "$EA_EXE" 2>/dev/null || echo 0) -lt 1000000 ]; then\n    wget -q https://origin-a.akamaihd.net/EA-Desktop-Client-Download/installer-releases/EAappInstaller.exe -O "$EA_EXE" || true\nfi\n\n# 11. Configure unprivileged gamer user\nid -u gamer >/dev/null 2>&1 || useradd -m -s /bin/bash -G audio,video gamer\nfor sub in .config .local .steam game-builds logs ea-prefix; do\n    mkdir -p "/home/gamer/$sub"\ndone\n\n# 12. Wine-GE runner installation for Heroic & Lutris\nWINE_GE_DIR="/home/gamer/.config/heroic/tools/wine"\nmkdir -p "$WINE_GE_DIR"\nWINE_GE_TAR="/tmp/wine-lutris-GE-Proton8-26-x86_64.tar.xz"\nif [ ! -d "$WINE_GE_DIR/Wine-GE-Proton8-26" ] && [ ! -d "$WINE_GE_DIR/lutris-GE-Proton8-26-x86_64" ]; then\n    echo "▶ Downloading Wine-GE-Proton8-26 compatibility runner..."\n    wget -q -c https://github.com/GloriousEggroll/wine-ge-custom/releases/download/GE-Proton8-26/wine-lutris-GE-Proton8-26-x86_64.tar.xz -O "$WINE_GE_TAR" || true\n    if [ -f "$WINE_GE_TAR" ] && [ $(stat -c%s "$WINE_GE_TAR") -gt 50000000 ]; then\n        echo "▶ Extracting Wine-GE-Proton8-26..."\n        tar -xf "$WINE_GE_TAR" -C "$WINE_GE_DIR" || true\n        for extracted in "$WINE_GE_DIR"/*; do\n            if [ -d "$extracted" ] && [ -f "$extracted/bin/wine" ]; then\n                ln -sf "$extracted" "$WINE_GE_DIR/Wine-GE-Proton8-26" || true\n                mkdir -p "/home/gamer/.local/share/lutris/runners/wine"\n                ln -sf "$extracted" "/home/gamer/.local/share/lutris/runners/wine/Wine-GE-Proton8-26" || true\n                if [ ! -f /usr/bin/wine ]; then\n                    ln -sf "$extracted/bin/wine" /usr/bin/wine || true\n                    ln -sf "$extracted/bin/wineserver" /usr/bin/wineserver || true\n                    ln -sf "$extracted/bin/wineboot" /usr/bin/wineboot || true\n                fi\n                if [ ! -f /usr/local/bin/wine ]; then\n                    ln -sf "$extracted/bin/wine" /usr/local/bin/wine || true\n                fi\n                break\n            fi\n        done\n    fi\nfi\n\n# 13. Pre-configure Heroic default settings: disable DXVK/VKD3D/Esync/Fsync\npython3 "$BENCH/fix_heroic_config.py" || true\n\n# 14. Pre-initialize clean 64-bit Wine prefix with Wine-GE (prevents kernel32.dll not found and popup dialogs)\nif [ -x "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wineboot" ]; then\n    echo "▶ Initializing clean default Wine prefix (/home/gamer/.wine)..."\n    export WINEPREFIX="/home/gamer/.wine"\n    export WINEARCH=win64\n    export WINEDLLOVERRIDES="mscoree,mshtml=d"\n    runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n        "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wineboot" --init 2>/dev/null || true\nfi\n\n# 14. Inject keyboard lock into Selkies HTML (prevents Esc from exiting browser fullscreen)\npython3 -c "\nimport sys; sys.path.insert(0, \'$BENCH\')\ntry:\n    from auto_optimize import inject_keyboard_lock\n    result = inject_keyboard_lock()\n    if result: print(\'Keyboard lock injected into:\', result)\n    else: print(\'No Selkies HTML found to patch for keyboard lock.\')\nexcept Exception as e: print(\'Keyboard lock injection skipped:\', e)\n" || true\n\n# 14. Compile Selkies gamepad interposers if not present\nif [ ! -f "$BENCH/lib/x86_64-linux-gnu/selkies_input_interposer.so" ]; then\n    SOURCE="$BENCH/selkies-source"\n    if [ ! -d "$SOURCE" ]; then\n        git clone --depth 1 --branch 2.1.0 https://github.com/selkies-project/selkies.git "$SOURCE" || true\n    fi\n    if [ -d "$SOURCE/addons/input-interposer" ]; then\n        make -C "$SOURCE/addons/input-interposer" PREFIX="$BENCH" install || true\n        make -C "$SOURCE/addons/input-interposer" PREFIX="$BENCH" install32 2>/dev/null || true\n        make -C "$SOURCE/addons/fake-udev" || true\n        make -C "$SOURCE/addons/fake-udev" all32 2>/dev/null || true\n        mkdir -p "$BENCH/lib/x86_64-linux-gnu" "$BENCH/lib/i386-linux-gnu"\n        cp -f "$SOURCE/addons/fake-udev/libudev.so.1" "$BENCH/lib/x86_64-linux-gnu/selkies_fake_udev.so" 2>/dev/null || true\n        cp -f "$SOURCE/addons/fake-udev/libudev_x86.so.1" "$BENCH/lib/i386-linux-gnu/selkies_fake_udev.so" 2>/dev/null || true\n    fi\nfi\n\n# 15. Set permissions and ownership\nchmod -R 755 "$BENCH"/*.sh 2>/dev/null || true\nchmod -R 777 "$BENCH" 2>/dev/null || true\nchown -R gamer:gamer /home/gamer 2>/dev/null || true\nchmod -R 777 /home/gamer 2>/dev/null || true\n\necho "=========================================================="\necho "    BENCH ENVIRONMENT SETUP COMPLETED SUCCESSFULLY!       "\necho "=========================================================="\n', encoding='utf-8')
else:
    print('▶ Updating bench repository from git...', flush=True)
    subprocess.run(['git', '-C', str(BENCH), 'reset', '--hard', 'HEAD'], check=False, timeout=30)
    subprocess.run(['git', '-C', str(BENCH), 'clean', '-fd'], check=False, timeout=30)
    subprocess.run(['git', '-C', str(BENCH), 'pull', '--force'], check=False, timeout=30)

# 3. Execute setup.sh with live output streaming
setup_sh.chmod(0o755)
print('▶ Running automated setup.sh...', flush=True)
proc = subprocess.Popen(['/bin/bash', str(setup_sh)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'setup.sh completed with exit code: {proc.returncode}', flush=True)

# 4. Configure unprivileged gamer user
subprocess.run(['useradd', '-m', '-s', '/bin/bash', '-G', 'audio,video', 'gamer'], check=False)
gamer_home = Path('/home/gamer')
for sub in ['.config', '.local', '.steam', 'game-builds', 'logs', 'ea-prefix']:
    (gamer_home / sub).mkdir(parents=True, exist_ok=True)

# Grant open permissions and ensure gamer ownership recursively
subprocess.run(['chown', '-R', 'gamer:gamer', str(gamer_home)], check=False)
subprocess.run(['chmod', '-R', '777', str(gamer_home)], check=False)
subprocess.run(['chmod', '-R', '777', str(BENCH)], check=False)

# 5. Storefront verification audit
def check_paths(candidates):
    for c in candidates:
        if c and os.path.exists(c):
            return c
        p = shutil.which(c)
        if p:
            return p
    return None

wine_ge_dir = gamer_home / '.config' / 'heroic' / 'tools' / 'wine'
storefront_status = {
    'steam': check_paths(['/usr/games/steam', '/usr/bin/steam']),
    'heroic': check_paths(['/usr/bin/heroic', '/opt/Heroic/heroic']),
    'legendary': check_paths(['/usr/local/bin/legendary', '/usr/bin/legendary', 'legendary']),
    'minigalaxy': check_paths(['/usr/games/minigalaxy', '/usr/bin/minigalaxy']),
    'lutris': check_paths(['/usr/games/lutris', '/usr/bin/lutris']),
    'wine': check_paths(['/usr/bin/wine', 'wine']),
    'wine_ge': check_paths([str(wine_ge_dir / 'Wine-GE-Proton8-26/bin/wine'), *[str(p) for p in wine_ge_dir.glob('*/bin/wine')]]),
    'ea_installer': str(EA_STORE / 'EAappInstaller.exe') if (EA_STORE / 'EAappInstaller.exe').exists() else None,
    'interposer_64': str(BENCH / 'lib/x86_64-linux-gnu/selkies_input_interposer.so') if (BENCH / 'lib/x86_64-linux-gnu/selkies_input_interposer.so').exists() else None,
    'interposer_32': str(BENCH / 'lib/i386-linux-gnu/selkies_input_interposer.so') if (BENCH / 'lib/i386-linux-gnu/selkies_input_interposer.so').exists() else None
}

print('================ STOREFRONT INSTALLATION AUDIT ================')
for k, v in storefront_status.items():
    print(f'  {k:15}: {v if v else "NOT FOUND / PENDING"}')
print('================================================================')


In [ ]:
from pathlib import Path
import os, subprocess

BENCH = Path('/tmp/kaggle-game-dev-bench')
BENCH.mkdir(parents=True, exist_ok=True)
(BENCH / 'logs').mkdir(parents=True, exist_ok=True)

# Write controller-friendly launcher UI
(BENCH / 'dev_launcher.py').write_text('"""Controller-friendly launcher and diagnostic interface for Linux gaming storefronts."""\nfrom pathlib import Path\nimport json\nimport math\nimport os\nimport shutil\nimport subprocess\nimport time\n\nimport pygame\n\n# Ensure /usr/games is in PATH\nos.environ[\'PATH\'] = f"/usr/games:/usr/local/bin:/usr/bin:{os.environ.get(\'PATH\', \'\')}"\n\nBENCH = Path(\'/tmp/kaggle-game-dev-bench\')\nLOGS = Path(\'/home/gamer/logs\')\nLOGS.mkdir(parents=True, exist_ok=True)\nBUILDS = Path(\'/tmp/game-builds\')\nBUILDS.mkdir(parents=True, exist_ok=True)\n\nWIDTH, HEIGHT = 1280, 720\nBG = (7, 15, 26)\nCARD_BG = (15, 28, 43)\nITEM_BG = (19, 38, 53)\nITEM_SEL_BG = (27, 68, 86)\nTEXT_WHITE = (235, 242, 255)\nTEXT_MUTED = (134, 154, 187)\nACCENT = (102, 207, 226)\nGREEN = (46, 204, 113)\nYELLOW = (241, 196, 15)\nRED = (231, 76, 60)\nBORDER_SEL = (77, 205, 242)\n\npygame.init()\npygame.display.set_caption(\'Game Lab\')\nscreen = pygame.display.set_mode((WIDTH, HEIGHT), pygame.RESIZABLE)\nclock = pygame.time.Clock()\n\nfont_title = pygame.font.SysFont(\'DejaVu Sans\', 27, bold=True)\nfont_hero = pygame.font.SysFont(\'DejaVu Sans\', 38, bold=True)\nfont_item = pygame.font.SysFont(\'DejaVu Sans\', 18, bold=True)\nfont_badge = pygame.font.SysFont(\'DejaVu Sans\', 13, bold=True)\nfont_body = pygame.font.SysFont(\'DejaVu Sans\', 15)\nfont_mono = pygame.font.SysFont(\'DejaVu Sans Mono\', 13)\nfont_small = pygame.font.SysFont(\'DejaVu Sans\', 13)\n\ndef resolve_bin(candidates):\n    for c in candidates:\n        if c and os.path.exists(c):\n            return c\n        p = shutil.which(c)\n        if p:\n            return p\n    return candidates[0] if candidates else None\n\nSTOREFRONTS = [\n    {\n        \'id\': \'optimizer\',\n        \'badge\': \'AUTO\',\n        \'badge_color\': (220, 160, 0),\n        \'title\': \'Auto-Optimize (تلقائي)\',\n        \'script\': str(BENCH / \'launch_optimizer.sh\'),\n        \'log\': str(LOGS / \'auto_optimize.log\'),\n        \'desc\': \'One-click auto-detect internet speed and apply optimal streaming & rendering quality. Also fixes Esc key exiting browser fullscreen.\',\n        \'compat\': \'Adjusts display resolution, Wine renderer, and keyboard lock automatically.\',\n        \'cmd\': \'python3 /tmp/kaggle-game-dev-bench/auto_optimize.py\'\n    },\n    {\n        \'id\': \'steam\',\n        \'badge\': \'STM\',\n        \'badge_color\': (30, 80, 160),\n        \'title\': \'Steam\',\n        \'script\': str(BENCH / \'launch_steam.sh\'),\n        \'log\': str(LOGS / \'steam.log\'),\n        \'desc\': \'Open Steam and sign in to your own library. The first launch may download client components and request agreement acceptance.\',\n        \'compat\': \'Linux native builds run directly. Proton/DXVK games requiring hardware Vulkan ICD will fail (Tesla T4 lacks container display/render nodes).\',\n        \'cmd\': \'steam -no-cef-sandbox -cef-disable-gpu -cef-force-software-rendering\'\n    },\n    {\n        \'id\': \'epic\',\n        \'badge\': \'EGS\',\n        \'badge_color\': (30, 140, 70),\n        \'title\': \'Epic Games · Heroic\',\n        \'script\': str(BENCH / \'launch_heroic.sh\'),\n        \'log\': str(LOGS / \'heroic.log\'),\n        \'desc\': \'Browse your Epic and GOG libraries in Heroic. For a game stuck on Launching, use the diagnostics tile and inspect its lastPlay log.\',\n        \'compat\': \'WineD3D is configured for Windows games. Actual game launch depends on the title and its runtime requirements.\',\n        \'cmd\': \'heroic --no-sandbox --disable-gpu --disable-gpu-compositing\'\n    },\n    {\n        \'id\': \'legendary\',\n        \'badge\': \'LEG\',\n        \'badge_color\': (35, 120, 180),\n        \'title\': \'Epic Games · Legendary\',\n        \'script\': str(BENCH / \'launch_legendary.sh\'),\n        \'log\': str(LOGS / \'legendary.log\'),\n        \'desc\': \'Direct Epic Games launch path for diagnosing Heroic. Sign in, list installed games, then launch with Wine and capture the exact output.\',\n        \'compat\': \'An earlier prototype launched Wolfenstein here. Recheck each title in the current Kaggle session.\',\n        \'cmd\': \'legendary launch APP --wine WINE_BIN\'\n    },\n    {\n        \'id\': \'heroic_fix\',\n        \'badge\': \'FIX\',\n        \'badge_color\': (220, 100, 30),\n        \'title\': \'Heroic diagnostics & repair\',\n        \'script\': str(BENCH / \'fix_heroic.sh\'),\n        \'log\': str(LOGS / \'heroic_fix.log\'),\n        \'desc\': \'Back up Heroic game settings, select WineD3D compatible options, and display recent launch logs. Existing Wine prefixes stay intact.\',\n        \'compat\': \'Settings repair can address runner mismatch. It does not guarantee every game will run.\',\n        \'cmd\': \'python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py\'\n    },\n    {\n        \'id\': \'gog\',\n        \'badge\': \'GOG\',\n        \'badge_color\': (140, 40, 160),\n        \'title\': \'GOG · Minigalaxy\',\n        \'script\': str(BENCH / \'launch_minigalaxy.sh\'),\n        \'log\': str(LOGS / \'minigalaxy.log\'),\n        \'desc\': "DRM-free GOG library client running as user \'gamer\'. Fast lightweight GTK interface for native Linux and Wine games. (GOG is also accessible inside Heroic).",\n        \'compat\': \'Client starts in this container; sign-in and individual game execution require interactive validation.\',\n        \'cmd\': \'minigalaxy\'\n    },\n    {\n        \'id\': \'lutris\',\n        \'badge\': \'LUT\',\n        \'badge_color\': (200, 120, 20),\n        \'title\': \'Lutris\',\n        \'script\': str(BENCH / \'launch_lutris.sh\'),\n        \'log\': str(LOGS / \'lutris.log\'),\n        \'desc\': "Universal game manager running as user \'gamer\'. Integrates Steam, Epic Games, GOG, EA, Wine runners, and custom Linux emulators and test builds.",\n        \'compat\': \'Multi-storefront aggregator. Runner management and automated installer scripts.\',\n        \'cmd\': \'lutris\'\n    },\n    {\n        \'id\': \'ea\',\n        \'badge\': \'EA \',\n        \'badge_color\': (180, 40, 40),\n        \'title\': \'EA App · experimental\',\n        \'script\': str(BENCH / \'launch_ea.sh\'),\n        \'log\': str(LOGS / \'ea.log\'),\n        \'desc\': "Electronic Arts Windows installer/client under Wine64 as user \'gamer\'. Downloads official EAappInstaller.exe into isolated wineprefix.",\n        \'compat\': \'Installer window was observed. Client sign-in and game launch are unverified.\',\n        \'cmd\': \'WINEPREFIX=/home/gamer/ea-prefix wine EAappInstaller.exe\'\n    },\n    {\n        \'id\': \'terminal\',\n        \'badge\': \'CLI\',\n        \'badge_color\': (80, 90, 110),\n        \'title\': \'Terminal (xterm)\',\n        \'script\': \'xterm\',\n        \'log\': str(LOGS / \'terminal.log\'),\n        \'desc\': \'Interactive X11 shell for inspecting the container, checking wine prefixes, running apt or git commands, or viewing logs.\',\n        \'compat\': \'Shell access in active Kaggle session.\',\n        \'cmd\': \'xterm -geometry 100x30 -title "Dev Bench Terminal"\'\n    },\n    {\n        \'id\': \'audio\',\n        \'badge\': \'AUD\',\n        \'badge_color\': (40, 160, 180),\n        \'title\': \'Stereo Audio Test\',\n        \'script\': None,\n        \'log\': None,\n        \'desc\': \'Generates stereo tone (440 Hz Left, 660 Hz Right) through PulseAudio monitor sink -> Selkies Opus stereo stream.\',\n        \'compat\': \'Verifies PulseAudio -> Selkies Opus 128 kbps stereo pipeline in your browser.\',\n        \'cmd\': \'pygame.mixer stereo tone (440Hz L / 660Hz R)\'\n    },\n    {\n        \'id\': \'video\',\n        \'badge\': \'VID\',\n        \'badge_color\': (60, 180, 100),\n        \'title\': \'Video Motion Test\',\n        \'script\': None,\n        \'log\': None,\n        \'desc\': \'Toggles real-time animated vertical bars to verify 60 FPS Xvfb capture and Tesla T4 NVENC H.264 encode smoothness.\',\n        \'compat\': \'Tesla T4 NVENC hardware encoder active.\',\n        \'cmd\': \'Real-time 60 FPS animated pattern\'\n    }\n]\n\n# Track background processes\nprocs = {}\n\ndef scan_builds():\n    builds = []\n    if BUILDS.exists():\n        for path in sorted(BUILDS.iterdir()):\n            if path.is_file() and path.suffix.lower() in (\'.sh\', \'.appimage\', \'.x86_64\', \'.bin\'):\n                builds.append({\n                    \'id\': f\'build_{path.name}\',\n                    \'badge\': \'BLD\',\n                    \'badge_color\': (100, 100, 200),\n                    \'title\': f\'Build: {path.name}\',\n                    \'script\': str(path),\n                    \'log\': str(LOGS / f\'{path.name}.log\'),\n                    \'desc\': f\'Uploaded Linux executable build in /tmp/game-builds/{path.name}. Upload more via Selkies Files panel.\',\n                    \'compat\': \'Executes inside container X11/PulseAudio environment.\',\n                    \'cmd\': f\'/bin/bash {path}\' if path.suffix.lower() == \'.sh\' else str(path)\n                })\n    return builds\n\ndef tone():\n    """A stereo left/right tone for testing the browser audio path."""\n    import array\n    rate = 48000\n    samples = array.array(\'h\')\n    for i in range(rate):\n        t = i / rate\n        samples.extend((int(7000 * math.sin(2 * math.pi * 440 * t)),\n                        int(7000 * math.sin(2 * math.pi * 660 * t))))\n    try:\n        pygame.mixer.init(frequency=rate, size=-16, channels=2, buffer=512)\n        pygame.mixer.Sound(buffer=samples.tobytes()).play()\n        return \'Stereo tone emitted: 440 Hz Left, 660 Hz Right\'\n    except Exception as exc:\n        return \'Audio unavailable: \' + str(exc)[:80]\n\ndef launch_item(item):\n    item_id = item[\'id\']\n    if item_id == \'audio\':\n        return tone()\n    if item_id == \'video\':\n        return \'Video test running in bottom strip. Check 60 FPS.\'\n    \n    # Check if already running\n    p = procs.get(item_id)\n    if p is not None and p.poll() is None:\n        return f\'{item["title"]} is already running (PID {p.pid}). Switch to it via taskbar or Alt+Tab.\'\n    \n    script = item.get(\'script\')\n    if not script:\n        return \'No executable script configured.\'\n    \n    try:\n        log_path = item.get(\'log\') or str(LOGS / f\'{item_id}.log\')\n        log_fh = open(log_path, \'a\', buffering=1)\n        log_fh.write(f"\\n--- Launching {item[\'title\']} at {time.strftime(\'%Y-%m-%d %H:%M:%S\')} ---\\n")\n        \n        env = os.environ.copy()\n        env[\'DISPLAY\'] = env.get(\'DISPLAY\', \':0\')\n        env[\'PATH\'] = f"/usr/games:/usr/local/bin:/usr/bin:{env.get(\'PATH\', \'\')}"\n        env[\'SDL_AUDIODRIVER\'] = \'pulse\'\n        \n        if script == \'xterm\':\n            cmd = [\'xterm\', \'-geometry\', \'100x30\', \'-title\', \'Dev Bench Terminal\']\n        elif script.endswith(\'.sh\'):\n            cmd = [\'/bin/bash\', script]\n        else:\n            if not os.access(script, os.X_OK):\n                return f\'{Path(script).name} is not executable. Set its execute bit after upload.\'\n            cmd = [script]\n            \n        proc = subprocess.Popen(cmd, stdout=log_fh, stderr=subprocess.STDOUT,\n                                stdin=subprocess.DEVNULL, env=env, start_new_session=True)\n        procs[item_id] = proc\n        return f\'Launched {item["title"]} (PID {proc.pid}). Window will appear in Openbox.\'\n    except Exception as exc:\n        return f\'Failed to launch {item["title"]}: {str(exc)[:80]}\'\n\ndef stop_item(item):\n    item_id = item[\'id\']\n    p = procs.get(item_id)\n    if p is not None and p.poll() is None:\n        try:\n            p.terminate()\n            return f\'Sent SIGTERM to {item["title"]} (PID {p.pid}).\'\n        except Exception as exc:\n            return f\'Error terminating process: {exc}\'\n    return f\'{item["title"]} is not currently running.\'\n\ndef get_proc_status(item):\n    item_id = item[\'id\']\n    if item_id in (\'audio\', \'video\'):\n        return \'READY\', GREEN\n    p = procs.get(item_id)\n    if p is None:\n        return \'IDLE\', TEXT_MUTED\n    poll = p.poll()\n    if poll is None:\n        return f\'RUNNING (PID {p.pid})\', GREEN\n    elif poll == 0:\n        return \'STOPPED (Exit 0)\', TEXT_MUTED\n    else:\n        return f\'EXITED ({poll})\', RED\n\ndef read_tail(log_path, lines=4):\n    if not log_path or not os.path.exists(log_path):\n        return [\'(No log entries yet)\']\n    try:\n        content = Path(log_path).read_text(encoding=\'utf-8\', errors=\'replace\').splitlines()\n        tail = [l for l in content if l.strip()][-lines:]\n        return tail if tail else [\'(Log is empty)\']\n    except Exception:\n        return [\'(Unable to read log)\']\n\nselected = 0\nmessage = \'Use D-Pad / Stick / Arrow keys to navigate. Press A / Enter to launch.\'\npointer_dx = pointer_dy = 0\npad_state = \'No gamepad event detected\'\nlast_scan = 0\nbuilds = []\nscroll_offset = 0\n\nrunning = True\nwhile running:\n    now = time.monotonic()\n    if now - last_scan > 2:\n        builds = scan_builds()\n        last_scan = now\n    \n    all_items = STOREFRONTS + builds\n    selected %= max(1, len(all_items))\n    cur = all_items[selected]\n    \n    # Event loop\n    for event in pygame.event.get():\n        if event.type == pygame.QUIT:\n            running = False\n        elif event.type == pygame.KEYDOWN:\n            if event.key in (pygame.K_ESCAPE, pygame.K_q):\n                running = False\n            elif event.key in (pygame.K_DOWN, pygame.K_s):\n                selected = (selected + 1) % len(all_items)\n            elif event.key in (pygame.K_UP, pygame.K_w):\n                selected = (selected - 1) % len(all_items)\n            elif event.key in (pygame.K_RETURN, pygame.K_SPACE):\n                message = launch_item(cur)\n            elif event.key in (pygame.K_BACKSPACE, pygame.K_DELETE):\n                message = stop_item(cur)\n            elif event.key == pygame.K_x:\n                message = tone()\n            elif pygame.K_1 <= event.key <= pygame.K_8:\n                idx = event.key - pygame.K_1\n                if idx < len(all_items):\n                    selected = idx\n                    message = launch_item(all_items[selected])\n        elif event.type == pygame.MOUSEMOTION:\n            pointer_dx, pointer_dy = event.rel\n        elif event.type == pygame.MOUSEBUTTONDOWN:\n            mx, my = event.pos\n            if event.button == 1:\n                # Left pane item click\n                if 30 <= mx <= 520 and 125 <= my <= 620:\n                    idx = (my - 125) // 56 + scroll_offset\n                    if 0 <= idx < len(all_items):\n                        selected = idx\n                        message = launch_item(all_items[selected])\n                # Launch button\n                elif 550 <= mx <= 700 and 560 <= my <= 605:\n                    message = launch_item(cur)\n                # Stop button\n                elif 715 <= mx <= 840 and 560 <= my <= 605:\n                    message = stop_item(cur)\n                # Audio button\n                elif 855 <= mx <= 1000 and 560 <= my <= 605:\n                    message = tone()\n            elif event.button == 4:\n                scroll_offset = max(0, scroll_offset - 1)\n            elif event.button == 5:\n                scroll_offset = min(max(0, len(all_items) - 8), scroll_offset + 1)\n        elif event.type == pygame.JOYDEVICEADDED:\n            pad = pygame.joystick.Joystick(event.device_index)\n            pad.init()\n            pad_state = f\'Gamepad Connected: {pad.get_name()[:40]}\'\n        elif event.type == pygame.JOYBUTTONDOWN:\n            pad_state = f\'Button {event.button} pressed\'\n            if event.button == 0:\n                message = launch_item(cur)\n            elif event.button == 1:\n                message = stop_item(cur)\n            elif event.button == 2:\n                message = tone()\n            elif event.button in (11, 13):\n                selected = (selected + (1 if event.button == 13 else -1)) % len(all_items)\n        elif event.type == pygame.JOYAXISMOTION:\n            if event.axis == 1:\n                if event.value > 0.6:\n                    selected = (selected + 1) % len(all_items)\n                    time.sleep(0.12)\n                elif event.value < -0.6:\n                    selected = (selected - 1) % len(all_items)\n                    time.sleep(0.12)\n            pad_state = f\'Axis {event.axis}: {event.value:+.2f}\'\n\n    # Adjust scroll offset\n    if selected < scroll_offset:\n        scroll_offset = selected\n    elif selected >= scroll_offset + 8:\n        scroll_offset = selected - 7\n\n    # Render frame\n    w, h = screen.get_size()\n    screen.fill(BG)\n    for gy in range(0, min(h, 720), 3):\n        t = gy / 720\n        pygame.draw.rect(screen, (int(7 + 10*t), int(15 + 17*t), int(26 + 21*t)), (0, gy, w, 3))\n\n    # Top Header Banner\n    pygame.draw.rect(screen, (20, 41, 55), pygame.Rect(0, 0, w, 112))\n    pygame.draw.rect(screen, ACCENT, pygame.Rect(0, 0, 7, 112))\n    screen.blit(font_title.render(\'GAME LAB\', True, TEXT_WHITE), (34, 16))\n    screen.blit(font_body.render(\'LIBRARY    •    STORES    •    BUILDS    •    DIAGNOSTICS\', True, ACCENT), (242, 29))\n    screen.blit(font_small.render(\'Temporary interactive game development session\', True, TEXT_MUTED), (36, 54))\n    \n    # Status pills in header\n    pills = [\n        (\'T4 NVENC  •  H.264\', GREEN),\n        (\'3D  •  llvmpipe\', YELLOW),\n        (\'AUDIO  •  OPUS\', GREEN),\n        (\'SESSION  •  gamer\', GREEN)\n    ]\n    px = 36\n    for ptext, pcolor in pills:\n        s_surf = font_small.render(ptext, True, pcolor)\n        pr = pygame.Rect(px, 78, s_surf.get_width() + 14, 24)\n        pygame.draw.rect(screen, CARD_BG, pr, border_radius=6)\n        pygame.draw.rect(screen, pcolor, pr, 1, border_radius=6)\n        screen.blit(s_surf, (px + 7, 82))\n        px += pr.width + 10\n\n    screen.blit(font_small.render(\'YOUR LIBRARY\', True, TEXT_MUTED), (36, 111))\n    # Left Pane: Storefront & App List\n    visible_items = all_items[scroll_offset:scroll_offset + 8]\n    for i, it in enumerate(visible_items):\n        actual_idx = scroll_offset + i\n        y = 125 + i * 56\n        is_sel = (actual_idx == selected)\n        rect = pygame.Rect(30, y, 480, 50)\n        \n        pygame.draw.rect(screen, ITEM_SEL_BG if is_sel else ITEM_BG, rect, border_radius=10)\n        if is_sel:\n            pygame.draw.rect(screen, BORDER_SEL, rect, 2, border_radius=10)\n            \n        badge_rect = pygame.Rect(40, y + 10, 42, 30)\n        pygame.draw.rect(screen, it[\'badge_color\'], badge_rect, border_radius=6)\n        b_txt = font_badge.render(it[\'badge\'], True, TEXT_WHITE)\n        screen.blit(b_txt, (badge_rect.centerx - b_txt.get_width() // 2, badge_rect.centery - b_txt.get_height() // 2))\n        \n        t_color = TEXT_WHITE if is_sel else (200, 215, 235)\n        screen.blit(font_item.render(it[\'title\'][:32], True, t_color), (92, y + 14))\n        \n        pstatus, pcolor = get_proc_status(it)\n        dot_color = GREEN if \'RUNNING\' in pstatus or pstatus == \'READY\' else (RED if \'EXITED\' in pstatus else TEXT_MUTED)\n        pygame.draw.circle(screen, dot_color, (495, y + 25), 5)\n\n    # Right Pane: Selected Item Detail Card\n    card_rect = pygame.Rect(530, 125, 720, 490)\n    pygame.draw.rect(screen, CARD_BG, card_rect, border_radius=14)\n    pygame.draw.rect(screen, (30, 45, 75), card_rect, 1, border_radius=14)\n    pygame.draw.rect(screen, cur[\'badge_color\'], pygame.Rect(530, 125, 720, 5), border_top_left_radius=14, border_top_right_radius=14)\n\n    # Card Title & Badge\n    cb_rect = pygame.Rect(550, 145, 50, 32)\n    pygame.draw.rect(screen, cur[\'badge_color\'], cb_rect, border_radius=6)\n    cb_txt = font_badge.render(cur[\'badge\'], True, TEXT_WHITE)\n    screen.blit(cb_txt, (cb_rect.centerx - cb_txt.get_width() // 2, cb_rect.centery - cb_txt.get_height() // 2))\n    screen.blit(font_title.render(cur[\'title\'], True, TEXT_WHITE), (610, 144))\n\n    # Description\n    desc_y = 190\n    words = cur[\'desc\'].split()\n    line = \'\'\n    for word in words:\n        test_line = line + (\' \' if line else \'\') + word\n        if font_body.size(test_line)[0] > 670:\n            screen.blit(font_body.render(line, True, (210, 225, 245)), (550, desc_y))\n            desc_y += 22\n            line = word\n        else:\n            line = test_line\n    if line:\n        screen.blit(font_body.render(line, True, (210, 225, 245)), (550, desc_y))\n        desc_y += 26\n\n    # Process Status Box\n    pbox_rect = pygame.Rect(550, desc_y, 680, 50)\n    pygame.draw.rect(screen, (13, 20, 35), pbox_rect, border_radius=8)\n    pstatus, pcolor = get_proc_status(cur)\n    screen.blit(font_small.render(\'PROCESS STATE:\', True, TEXT_MUTED), (562, desc_y + 8))\n    screen.blit(font_item.render(pstatus, True, pcolor), (562, desc_y + 24))\n    desc_y += 62\n\n    # Command line\n    screen.blit(font_small.render(\'EXECUTION COMMAND:\', True, TEXT_MUTED), (550, desc_y))\n    desc_y += 18\n    cmd_rect = pygame.Rect(550, desc_y, 680, 32)\n    pygame.draw.rect(screen, (13, 20, 35), cmd_rect, border_radius=6)\n    screen.blit(font_mono.render(cur[\'cmd\'][:85], True, ACCENT), (560, desc_y + 8))\n    desc_y += 44\n\n    # Compatibility Box\n    screen.blit(font_small.render(\'COMPATIBILITY & HARDWARE DIAGNOSTICS:\', True, TEXT_MUTED), (550, desc_y))\n    desc_y += 18\n    compat_rect = pygame.Rect(550, desc_y, 680, 52)\n    pygame.draw.rect(screen, (13, 20, 35), compat_rect, border_radius=6)\n    screen.blit(font_small.render(cur[\'compat\'][:95], True, YELLOW), (560, desc_y + 8))\n    screen.blit(font_small.render(\'T4 NVENC captures screen. Games requiring hardware Vulkan/DXVK will fail without ICD.\', True, TEXT_MUTED), (560, desc_y + 28))\n    desc_y += 64\n\n    # Log tail\n    screen.blit(font_small.render(\'RECENT LOG OUTPUT:\', True, TEXT_MUTED), (550, desc_y))\n    desc_y += 18\n    log_rect = pygame.Rect(550, desc_y, 680, 78)\n    pygame.draw.rect(screen, (10, 15, 26), log_rect, border_radius=6)\n    tails = read_tail(cur.get(\'log\'), lines=4)\n    for ly, line_txt in enumerate(tails):\n        screen.blit(font_mono.render(line_txt[:90], True, (170, 185, 205)), (560, desc_y + 6 + ly * 17))\n    desc_y += 90\n\n    # Action Buttons\n    btn_launch = pygame.Rect(550, 560, 150, 42)\n    pygame.draw.rect(screen, (30, 110, 180), btn_launch, border_radius=8)\n    screen.blit(font_item.render(\'▶ Launch (A)\', True, TEXT_WHITE), (568, 571))\n    \n    btn_stop = pygame.Rect(715, 560, 130, 42)\n    pygame.draw.rect(screen, (140, 40, 40), btn_stop, border_radius=8)\n    screen.blit(font_item.render(\'■ Stop (B)\', True, TEXT_WHITE), (735, 571))\n    \n    btn_audio = pygame.Rect(860, 560, 150, 42)\n    pygame.draw.rect(screen, (30, 120, 90), btn_audio, border_radius=8)\n    screen.blit(font_item.render(\'♫ Audio (X)\', True, TEXT_WHITE), (878, 571))\n\n    screen.blit(font_small.render(\'D-Pad/Stick: Navigate\', True, TEXT_MUTED), (1030, 564))\n    screen.blit(font_small.render(\'Enter: Launch\', True, TEXT_MUTED), (1030, 584))\n\n    # Bottom Area: Motion test bar + Diagnostics footer\n    bottom_y = 625\n    tick = now * 180\n    for i in range(16):\n        bx = int((tick + i * 85) % (w + 100)) - 50\n        pygame.draw.rect(screen, (20 + i * 4, 45 + i * 5, 80 + i * 6),\n                         pygame.Rect(bx, bottom_y, 25, 6), border_radius=3)\n    \n    # Status Row\n    screen.blit(font_small.render(f\'FPS: {clock.get_fps():.1f} | Display: {w}x{h} | Mouse: {pointer_dx:+d}, {pointer_dy:+d}\', True, TEXT_WHITE), (34, 638))\n    screen.blit(font_small.render(pad_state[:75], True, TEXT_MUTED), (34, 660))\n    screen.blit(font_small.render(message[:110], True, ACCENT), (34, 684))\n\n    pygame.display.flip()\n    clock.tick(60)\n\npygame.quit()\n', encoding='utf-8')

# Write tint2 panel configuration for desktop taskbar
tint2rc = """# Minimal dark tint2 panel configuration
panel_position = bottom center horizontal
panel_size = 100% 32
panel_margin = 0 0
panel_padding = 4 2 4
panel_background_id = 1
wm_menu = 1
panel_dock = 0
panel_layer = top

taskbar_mode = single_desktop
taskbar_padding = 2 0 2
taskbar_background_id = 0
taskbar_active_background_id = 2

task_icon = 1
task_text = 1
task_width = 190
task_maximum_size = 220 28
task_padding = 6 2
task_font = DejaVu Sans 10
task_font_color = #d8e4f5 100
task_active_font_color = #4dcdf2 100

time1_font = DejaVu Sans 10
clock_font_color = #9bdcf5 100
time1_format = %H:%M:%S

rounded = 0
border_width = 1
background_color = #090e1e 100
border_color = #172b4d 100

rounded = 4
border_width = 1
background_color = #152744 100
border_color = #4dcdf2 100
"""
(BENCH / 'tint2rc').write_text(tint2rc, encoding='utf-8')

# Steam launch wrapper - executes under unprivileged 'gamer' user
steam_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export SDL_AUDIODRIVER=pulse
export LD_PRELOAD='/tmp/kaggle-game-dev-bench/$LIB/selkies_input_interposer.so:/tmp/kaggle-game-dev-bench/$LIB/selkies_fake_udev.so'
export SELKIES_JS_SOCKET_PATH="$XDG_RUNTIME_DIR"
export SDL_JOYSTICK_DISABLE_UDEV=1
export SDL_JOYSTICK_DEVICE=/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003

mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/steam.log"
STEAM_BIN=$(which steam 2>/dev/null || echo /usr/games/steam)

echo "Starting Steam as user gamer at $(date)..." >> "$LOG"
exec env \
    DISPLAY="$DISPLAY" \
    PATH="$PATH" \
    SDL_AUDIODRIVER=pulse \
    LD_PRELOAD="$LD_PRELOAD" \
    SELKIES_JS_SOCKET_PATH="$SELKIES_JS_SOCKET_PATH" \
    SDL_JOYSTICK_DISABLE_UDEV=1 \
    SDL_JOYSTICK_DEVICE="$SDL_JOYSTICK_DEVICE" \
    HOME="/home/gamer" \
    "$STEAM_BIN" -no-cef-sandbox -cef-disable-gpu -cef-force-software-rendering "$@" >> "$LOG" 2>&1
"""
(BENCH / 'launch_steam.sh').write_text(steam_sh, encoding='utf-8')

# Heroic launch wrapper (Epic Games & GOG) - executes under unprivileged 'gamer' user
heroic_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export SDL_AUDIODRIVER=pulse
export LD_PRELOAD='/tmp/kaggle-game-dev-bench/$LIB/selkies_input_interposer.so:/tmp/kaggle-game-dev-bench/$LIB/selkies_fake_udev.so'
export SELKIES_JS_SOCKET_PATH="$XDG_RUNTIME_DIR"
export SDL_JOYSTICK_DISABLE_UDEV=1
export SDL_JOYSTICK_DEVICE=/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003

mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/heroic.log"
HEROIC_BIN=$(which heroic 2>/dev/null || echo /usr/bin/heroic)

# Repair settings with backups and leave existing Wine prefixes intact.
python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py >> "$LOG" 2>&1

echo "Starting Heroic as user gamer at $(date)..." >> "$LOG"
exec env \
    DISPLAY="$DISPLAY" \
    PATH="$PATH" \
    SDL_AUDIODRIVER=pulse \
    LD_PRELOAD="$LD_PRELOAD" \
    SELKIES_JS_SOCKET_PATH="$SELKIES_JS_SOCKET_PATH" \
    SDL_JOYSTICK_DISABLE_UDEV=1 \
    SDL_JOYSTICK_DEVICE="$SDL_JOYSTICK_DEVICE" \
    HOME="/home/gamer" \
    "$HEROIC_BIN" --no-sandbox --disable-gpu --disable-gpu-compositing "$@" >> "$LOG" 2>&1
"""
(BENCH / 'launch_heroic.sh').write_text(heroic_sh, encoding='utf-8')

# Minigalaxy launch wrapper (GOG dedicated) - executes under unprivileged 'gamer' user
mini_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export SDL_AUDIODRIVER=pulse
mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/minigalaxy.log"
MINI_BIN=$(which minigalaxy 2>/dev/null || echo /usr/games/minigalaxy)

echo "Starting Minigalaxy as user gamer at $(date)..." >> "$LOG"
exec env \
    DISPLAY="$DISPLAY" \
    PATH="$PATH" \
    SDL_AUDIODRIVER=pulse \
    HOME="/home/gamer" \
    "$MINI_BIN" "$@" >> "$LOG" 2>&1
"""
(BENCH / 'launch_minigalaxy.sh').write_text(mini_sh, encoding='utf-8')

# Lutris launch wrapper - executes under unprivileged 'gamer' user
lutris_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export SDL_AUDIODRIVER=pulse
mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/lutris.log"
LUTRIS_BIN=$(which lutris 2>/dev/null || echo /usr/games/lutris)

# Match Ubuntu noble C extension ABI (_dbus_bindings) using python3.12 if available
if [ -x /usr/bin/python3.12 ]; then
    PY_BIN="/usr/bin/python3.12"
else
    PY_BIN="python3"
fi

echo "Starting Lutris with $PY_BIN as user gamer at $(date)..." >> "$LOG"
exec env \
    DISPLAY="$DISPLAY" \
    PATH="$PATH" \
    SDL_AUDIODRIVER=pulse \
    HOME="/home/gamer" \
    "$PY_BIN" "$LUTRIS_BIN" "$@" >> "$LOG" 2>&1
"""
(BENCH / 'launch_lutris.sh').write_text(lutris_sh, encoding='utf-8')

# EA App launch wrapper - executes under unprivileged 'gamer' user in isolated wineprefix
ea_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export WINEPREFIX="/home/gamer/ea-prefix"
export WINEARCH=win64
export WINEDLLOVERRIDES="mscoree="
mkdir -p "$WINEPREFIX" /tmp/ea-store /home/gamer/logs
chown -R gamer:gamer "$WINEPREFIX" /home/gamer/logs 2>/dev/null || true

LOG="/home/gamer/logs/ea.log"
echo "Starting EA App runner at $(date)..." >> "$LOG"

EA_INSTALLER="/tmp/ea-store/EAappInstaller.exe"
if [ ! -f "$EA_INSTALLER" ]; then
    echo "Downloading EAappInstaller.exe..." >> "$LOG"
    wget -q https://origin-a.akamaihd.net/EA-Desktop-Client-Download/installer-releases/EAappInstaller.exe -O "$EA_INSTALLER"
fi

EA_INSTALLED="$WINEPREFIX/drive_c/Program Files/Electronic Arts/EA Desktop/EA Desktop/EADesktop.exe"
if [ -f "$EA_INSTALLED" ]; then
    echo "Running installed EA Desktop: $EA_INSTALLED" >> "$LOG"
    TARGET="$EA_INSTALLED"
else
    echo "Running EA App Installer via Wine64: $EA_INSTALLER" >> "$LOG"
    TARGET="$EA_INSTALLER"
fi

exec env \
    DISPLAY="$DISPLAY" \
    PATH="$PATH" \
    WINEPREFIX="$WINEPREFIX" \
    WINEARCH=win64 \
    WINEDLLOVERRIDES="$WINEDLLOVERRIDES" \
    HOME="/home/gamer" \
    wine "$TARGET" "$@" >> "$LOG" 2>&1
"""
(BENCH / 'launch_ea.sh').write_text(ea_sh, encoding='utf-8')

# Fix Heroic Config Python script
(BENCH / 'fix_heroic_config.py').write_text('"""Apply reversible Heroic settings for this container, disarm DXVK, and sync Legendary auth."""\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nfrom datetime import datetime, timezone\n\nHOME = Path(os.environ.get(\'KAGGLE_GAMER_HOME\', \'/home/gamer\'))\nCONFIG = HOME / \'.config/heroic\'\nGAME_CONFIG = CONFIG / \'GamesConfig\'\nRUNNERS = CONFIG / \'tools/wine\'\nBACKUPS = HOME / \'logs/heroic-config-backups\'\nBACKUPS.mkdir(parents=True, exist_ok=True)\n\n# Select best available Wine runner (Prefer Wine-GE over system wine)\nrunner = {\'bin\': \'/usr/bin/wine\', \'wineserver\': \'/usr/bin/wineserver\',\n          \'wineboot\': \'/usr/bin/wineboot\', \'name\': \'Wine Default\', \'type\': \'wine\'}\nfor candidate in [RUNNERS / \'Wine-GE-Proton8-26/bin/wine\', *RUNNERS.glob(\'*/bin/wine\')]:\n    if (candidate.is_file() and os.access(candidate, os.X_OK) and\n            candidate.with_name(\'wineserver\').is_file()):\n        runner = {\'bin\': str(candidate), \'wineserver\': str(candidate.with_name(\'wineserver\')),\n                  \'wineboot\': str(candidate.with_name(\'wineboot\')),\n                  \'name\': candidate.parent.parent.name, \'type\': \'wine\'}\n        break\n\nGAME_SETTINGS_KEYS = (\n    \'autoInstallDxvk\', \'wineVersion\', \'winePrefix\', \'autoInstallVkd3d\',\n    \'preferSystemLibs\', \'launcherArgs\', \'enableEsync\', \'enableFsync\'\n)\n\ndef clean_prefix_dxvk(prefix: Path):\n    """Safely disarm DXVK DLL overrides so WineD3D (OpenGL) renders without Vulkan hardware crash."""\n    for sub in (\'drive_c/windows/system32\', \'drive_c/windows/syswow64\'):\n        sys_dir = prefix / sub\n        if not sys_dir.is_dir():\n            continue\n        for dll_name in (\'dxgi.dll\', \'d3d11.dll\', \'d3d10core.dll\', \'d3d9.dll\'):\n            target = sys_dir / dll_name\n            if target.is_file():\n                try:\n                    bak = target.with_name(target.name + \'.dxvk_bak\')\n                    if not bak.exists():\n                        target.rename(bak)\n                    else:\n                        target.unlink()\n                    print(f\'Disarmed DXVK dll in prefix: {target}\')\n                except OSError as err:\n                    print(f\'Notice disarming {target}: {err}\')\n\ndef repair(path, global_config=False):\n    if not path.exists():\n        return \'missing\'\n    try:\n        old_text = path.read_text(encoding=\'utf-8\')\n        data = json.loads(old_text)\n        if not isinstance(data, dict):\n            return \'unexpected JSON shape\'\n    except (OSError, json.JSONDecodeError) as exc:\n        return f\'not edited: {exc}\'\n\n    settings_objects = []\n    if global_config:\n        # Migrate flat config if found\n        if not isinstance(data.get(\'defaultSettings\'), dict):\n            keys = (\'autoInstallDxvk\', \'autoInstallVkd3d\', \'autoInstallDxvkNvapi\',\n                    \'enableEsync\', \'enableFsync\', \'disableUMU\', \'wineVersion\')\n            data[\'defaultSettings\'] = {key: data.pop(key) for key in keys if key in data}\n        if isinstance(data.get(\'defaultSettings\'), dict):\n            settings_objects.append(data[\'defaultSettings\'])\n        else:\n            settings_objects.append(data)\n    else:\n        # Per-game config: handle stem key, arbitrary single key, multiple keys, or flat\n        if path.stem in data and isinstance(data[path.stem], dict):\n            settings_objects.append(data[path.stem])\n        else:\n            child_matches = [\n                v for v in data.values()\n                if isinstance(v, dict) and any(k in v for k in GAME_SETTINGS_KEYS)\n            ]\n            if child_matches:\n                settings_objects.extend(child_matches)\n            elif any(k in data for k in GAME_SETTINGS_KEYS):\n                settings_objects.append(data)\n            elif len(data) == 1 and isinstance(list(data.values())[0], dict):\n                settings_objects.append(list(data.values())[0])\n\n    if not settings_objects:\n        return \'unexpected settings shape\'\n\n    for settings in settings_objects:\n        for key in (\'autoInstallDxvk\', \'autoInstallVkd3d\', \'autoInstallDxvkNvapi\',\n                    \'enableEsync\', \'enableFsync\'):\n            settings[key] = False\n        settings[\'disableUMU\'] = True\n\n        # Assign Wine runner\n        selected = settings.get(\'wineVersion\')\n        if (not isinstance(selected, dict) or selected.get(\'type\') != \'wine\' or\n                not os.path.isfile(selected.get(\'bin\', \'\')) or\n                not os.path.isfile(selected.get(\'wineserver\', \'\'))):\n            settings[\'wineVersion\'] = runner\n\n        # Ensure WINEDLLOVERRIDES forces WineD3D builtins\n        env_opts = settings.get(\'enviromentOptions\')\n        if not isinstance(env_opts, list):\n            env_opts = []\n        found_override = False\n        for opt in env_opts:\n            if isinstance(opt, dict) and opt.get(\'key\') == \'WINEDLLOVERRIDES\':\n                opt[\'value\'] = \'d3d11,dxgi,d3d9,d3d10core=b\'\n                found_override = True\n                break\n        if not found_override:\n            env_opts.append({\'key\': \'WINEDLLOVERRIDES\', \'value\': \'d3d11,dxgi,d3d9,d3d10core=b\'})\n        settings[\'enviromentOptions\'] = env_opts\n\n        # Clean prefix DXVK overrides if prefix exists\n        pfx = settings.get(\'winePrefix\')\n        if pfx and os.path.isdir(pfx):\n            clean_prefix_dxvk(Path(pfx))\n\n    new_text = json.dumps(data, indent=2) + \'\\n\'\n    if new_text == old_text:\n        return \'already configured\'\n    stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\')\n    backup = BACKUPS / f\'{path.name}.{stamp}.bak\'\n    shutil.copy2(path, backup)\n    tmp = path.with_suffix(path.suffix + \'.new\')\n    tmp.write_text(new_text, encoding=\'utf-8\')\n    os.replace(tmp, path)\n    return f\'updated; backup {backup}\'\n\ndef sync_legendary():\n    """Seamlessly share Epic Games auth tokens & installed library between Heroic and Legendary CLI."""\n    heroic_leg = CONFIG / \'legendaryConfig/legendary\'\n    std_leg = HOME / \'.config/legendary\'\n    heroic_leg.mkdir(parents=True, exist_ok=True)\n    std_leg.mkdir(parents=True, exist_ok=True)\n\n    synced = []\n    for fname in (\'user.json\', \'installed.json\', \'config.ini\'):\n        h_f = heroic_leg / fname\n        s_f = std_leg / fname\n        try:\n            if h_f.is_file() and not s_f.is_file():\n                shutil.copy2(h_f, s_f)\n                synced.append(f\'{fname} (Heroic -> Legendary)\')\n            elif s_f.is_file() and not h_f.is_file():\n                shutil.copy2(s_f, h_f)\n                synced.append(f\'{fname} (Legendary -> Heroic)\')\n            elif h_f.is_file() and s_f.is_file():\n                if h_f.stat().st_mtime > s_f.stat().st_mtime:\n                    shutil.copy2(h_f, s_f)\n                    synced.append(f\'{fname} (Heroic -> Legendary, updated)\')\n                elif s_f.stat().st_mtime > h_f.stat().st_mtime:\n                    shutil.copy2(s_f, h_f)\n                    synced.append(f\'{fname} (Legendary -> Heroic, updated)\')\n        except Exception as e:\n            print(f\'Notice syncing {fname}: {e}\')\n    if synced:\n        print(\'Legendary/Heroic synced:\', \', \'.join(synced))\n    else:\n        print(\'Legendary/Heroic auth status: synchronized\')\n\nprint(\'Heroic runner:\', runner[\'bin\'])\nglobal_file = CONFIG / \'config.json\'\nprint(global_file, repair(global_file, global_config=True))\nfor path in sorted(GAME_CONFIG.glob(\'*.json\')):\n    print(path, repair(path))\nfor log in sorted(GAME_CONFIG.glob(\'*-lastPlay.log\'), key=lambda p: p.stat().st_mtime, reverse=True)[:5]:\n    print(f\'Latest play log: {log}\')\n    print(\'\\n\'.join(log.read_text(errors=\'replace\').splitlines()[-25:]))\n\nsync_legendary()\n', encoding='utf-8')

# Fix Heroic Shell wrapper for launcher
fix_heroic_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/heroic_fix.log"
exec xterm -geometry 95x25 -title "Heroic Config Sanitizer" -e bash -c "echo '=== Sanitizing Heroic Settings & Prefixes ===' | tee -a '$LOG'; python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py 2>&1 | tee -a '$LOG'; chown -R gamer:gamer /home/gamer 2>/dev/null || true; chmod -R u+rwX /home/gamer 2>/dev/null || true; echo ''; echo 'Sanitization complete! All games configured for WineD3D.'; echo ''; read -p 'Press Enter to close window...'"
"""
(BENCH / 'fix_heroic.sh').write_text(fix_heroic_sh, encoding='utf-8')

# Legendary interactive menu and graphical interface
(BENCH / 'legendary_menu.sh').write_text('#!/bin/bash\nexport DISPLAY="${DISPLAY:-:0}"\nexport PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"\nexport SDL_AUDIODRIVER=pulse\nexport HOME="/home/gamer"\nexport LEGENDARY_CONFIG_PATH="$HOME/.config/heroic/legendaryConfig/legendary"\nexport BROWSER="/bin/true"\nmkdir -p "$LEGENDARY_CONFIG_PATH" "$HOME/.config/legendary" "$HOME/logs"\nset -o pipefail\n\n# Source auto-optimizer Wine settings if available\nif [ -f "$HOME/.config/gamelab_wine_env.sh" ]; then\n    source "$HOME/.config/gamelab_wine_env.sh"\nfi\n\n# Ensure credentials and configs are synced on startup\npython3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py >/dev/null 2>&1 || true\n\nwhile true; do\n    clear\n    echo "=================================================================="\n    echo "           LEGENDARY CLI - EPIC GAMES NATIVE MANAGER              "\n    echo "=================================================================="\n    echo " Epic Library Location : $LEGENDARY_CONFIG_PATH"\n    echo " Rendering Backend     : WineD3D (OpenGL software rendering / llvmpipe)"\n    echo " Shared Session        : Synced automatically with Heroic Games"\n    echo "------------------------------------------------------------------"\n    echo " [1] List installed games and launchable apps"\n    echo " [2] Quick Launch an installed game (Brotato, Wolfenstein, etc)"\n    echo " [3] Log in to Epic Games (or check login status)"\n    echo " [4] List all games in your Epic Games library"\n    echo " [5] Install a game from your Epic library"\n    echo " [6] Fix Heroic settings (Disable DXVK / Enable WineD3D)"\n    echo " [7] Open an interactive Bash shell as gamer user"\n    echo " [8] Print a game\'s launch command without starting it"\n    echo " [0] Exit"\n    echo "=================================================================="\n    read -p "Select option [0-8]: " choice\n\n    case "$choice" in\n        1)\n            echo ""\n            echo "Syncing Epic library from Heroic..."\n            python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py 2>/dev/null || true\n            echo ""\n            legendary list-installed\n            echo ""\n            read -p "Press Enter to return to menu..."\n            ;;\n        2)\n            echo ""\n            echo "Installed games:"\n            legendary list-installed\n            echo ""\n            read -p "Enter Game App Name (from the list above): " appname\n            if [ -n "$appname" ]; then\n                WINE_BIN=""\n                for candidate in "/home/gamer/.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine" \\\n                                 /home/gamer/.config/heroic/tools/wine/*/bin/wine \\\n                                 /usr/bin/wine /usr/local/bin/wine /usr/games/wine; do\n                    if [ -f "$candidate" ] && [ -x "$candidate" ]; then\n                        WINE_BIN="$candidate"\n                        break\n                    fi\n                done\n                if [ -z "$WINE_BIN" ]; then\n                    echo "Error: No Wine runner found on system!"\n                else\n                    echo "Launching $appname via WineD3D using runner: $WINE_BIN"\n                    export WINEDLLOVERRIDES="d3d11,dxgi,d3d9,d3d10core=b;mscoree,mshtml=d"\n                    export WINEPREFIX="/home/gamer/.wine"\n                    \n                    # Auto-heal prefix if kernel32.dll is missing\n                    if [ ! -f "$WINEPREFIX/drive_c/windows/system32/kernel32.dll" ]; then\n                        echo "Initializing Wine prefix with Wine-GE..."\n                        WINEBOOT_BIN="$(dirname "$WINE_BIN")/wineboot"\n                        if [ -x "$WINEBOOT_BIN" ]; then\n                            WINEDLLOVERRIDES="mscoree,mshtml=d" "$WINEBOOT_BIN" --init 2>/dev/null || true\n                        fi\n                    fi\n                    \n                    legendary launch "$appname" --wine "$WINE_BIN" 2>&1 | tee -a /home/gamer/logs/legendary-game.log\n                    echo "Legendary exit status: ${PIPESTATUS[0]}" | tee -a /home/gamer/logs/legendary-game.log\n                fi\n            fi\n            read -p "Press Enter to return to menu..."\n            ;;\n        3)\n            echo ""\n            echo "Checking Epic Games login status..."\n            python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py 2>/dev/null || true\n            if [ -f "$HOME/.config/heroic/legendaryConfig/legendary/user.json" ] || [ -f "$HOME/.config/legendary/user.json" ]; then\n                echo "✓ Found existing Epic Games session!"\n                legendary status 2>/dev/null\n                echo ""\n                read -p "Already logged in! Do you want to re-authenticate with a new code? (y/N): " reauth\n                if [[ ! "$reauth" =~ ^[Yy]$ ]]; then\n                    read -p "Press Enter to return to menu..."\n                    continue\n                fi\n            fi\n\n            echo ""\n            echo "=================================================================="\n            echo "                   EPIC GAMES LOGIN INSTRUCTIONS                  "\n            echo "=================================================================="\n            echo " 1. Open this URL in your PC browser (Chrome/Edge/Firefox):"\n            echo "    https://legendary.gl/epiclogin"\n            echo ""\n            echo " 2. Sign in with your Epic Games account."\n            echo ""\n            echo " 3. The page will display a JSON code containing \'authorizationCode\'."\n            echo "    Example: {\\"authorizationCode\\": \\"3a7b9c1d2e...\\"}"\n            echo ""\n            echo " 4. Copy the authorizationCode value (or the whole JSON text)"\n            echo "    and paste it into the prompt below."\n            echo "=================================================================="\n            echo ""\n            legendary auth\n            python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py 2>/dev/null || true\n            echo ""\n            read -p "Press Enter to return to menu..."\n            ;;\n        4)\n            echo ""\n            legendary list\n            echo ""\n            read -p "Press Enter to return to menu..."\n            ;;\n        5)\n            echo ""\n            read -p "Enter Game App Name to install: " install_app\n            if [ -n "$install_app" ]; then\n                legendary install "$install_app"\n                python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py 2>/dev/null || true\n            fi\n            read -p "Press Enter to return to menu..."\n            ;;\n        6)\n            echo ""\n            python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py\n            echo ""\n            read -p "Press Enter to return to menu..."\n            ;;\n        7)\n            echo "Dropping into shell as user gamer. Type \'exit\' to return."\n            bash\n            ;;\n        8)\n            legendary list-installed\n            read -p "Enter Game App Name: " appname\n            if [ -n "$appname" ]; then\n                WINE_BIN=""\n                for candidate in "/home/gamer/.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine" \\\n                                 /home/gamer/.config/heroic/tools/wine/*/bin/wine \\\n                                 /usr/bin/wine /usr/local/bin/wine /usr/games/wine; do\n                    if [ -f "$candidate" ] && [ -x "$candidate" ]; then\n                        WINE_BIN="$candidate"\n                        break\n                    fi\n                done\n                legendary launch "$appname" --wine "${WINE_BIN:-wine}" --dry-run 2>&1 | tee -a /home/gamer/logs/legendary-dry-run.log\n                echo "Legendary dry-run exit status: ${PIPESTATUS[0]}" | tee -a /home/gamer/logs/legendary-dry-run.log\n            fi\n            read -p "Press Enter to return to menu..."\n            ;;\n        0)\n            break\n            ;;\n        *)\n            echo "Invalid option."\n            sleep 1\n            ;;\n    esac\ndone\n', encoding='utf-8')
(BENCH / 'legendary_gui.py').write_text('"""Modern Graphical Interface for Legendary Epic Games Manager."""\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nimport tkinter as tk\nfrom tkinter import ttk, messagebox, scrolledtext\n\nHOME = Path(os.environ.get(\'KAGGLE_GAMER_HOME\', \'/home/gamer\'))\nCONFIG = HOME / \'.config/heroic\'\nHEROIC_LEG = CONFIG / \'legendaryConfig/legendary\'\nSTD_LEG = HOME / \'.config/legendary\'\nBENCH = Path(\'/tmp/kaggle-game-dev-bench\')\nLOGS = HOME / \'logs\'\nLOGS.mkdir(parents=True, exist_ok=True)\n\n# Ensure config paths exist\nHEROIC_LEG.mkdir(parents=True, exist_ok=True)\nSTD_LEG.mkdir(parents=True, exist_ok=True)\n\ndef find_best_runner():\n    candidates = [\n        HOME / \'.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine\',\n        *HOME.glob(\'.config/heroic/tools/wine/*/bin/wine\'),\n        Path(\'/usr/bin/wine\'),\n        Path(\'/usr/local/bin/wine\'),\n        Path(\'/usr/games/wine\'),\n    ]\n    for c in candidates:\n        if c.is_file() and os.access(c, os.X_OK):\n            return str(c)\n    return \'/usr/bin/wine\'\n\ndef format_size(bytes_val):\n    try:\n        b = float(bytes_val)\n        if b >= 1024 * 1024 * 1024:\n            return f"{b / (1024**3):.2f} GiB"\n        elif b >= 1024 * 1024:\n            return f"{b / (1024**2):.1f} MiB"\n        elif b > 0:\n            return f"{b / 1024:.0f} KiB"\n    except (ValueError, TypeError):\n        pass\n    return "Unknown"\n\nclass LegendaryApp(tk.Tk):\n    def __init__(self):\n        super().__init__()\n        self.title("Epic Games · Legendary Manager")\n        self.geometry("980x660")\n        self.minsize(800, 500)\n\n        # Active process tracker\n        self.active_proc = None\n        self.proc_thread = None\n\n        # Apply dark theme styling\n        self.setup_styles()\n\n        # Build UI layout\n        self.create_header()\n        self.create_tabs()\n        self.create_bottom_bar()\n\n        # Initial data load\n        self.refresh_all()\n\n    def setup_styles(self):\n        self.configure(bg="#141923")\n        self.style = ttk.Style(self)\n        try:\n            self.style.theme_use("clam")\n        except tk.TclError:\n            pass\n\n        # Colors\n        self.c_bg = "#141923"\n        self.c_panel = "#1b2230"\n        self.c_card = "#222c3e"\n        self.c_fg = "#f0f4fc"\n        self.c_muted = "#8fa0bd"\n        self.c_accent = "#3498db"\n        self.c_green = "#2ecc71"\n        self.c_red = "#e74c3c"\n        self.c_yellow = "#f39c12"\n\n        self.style.configure(".", background=self.c_bg, foreground=self.c_fg, font=("DejaVu Sans", 10))\n        self.style.configure("TNotebook", background=self.c_bg, borderwidth=0)\n        self.style.configure("TNotebook.Tab", background=self.c_panel, foreground=self.c_muted,\n                             padding=[18, 9], font=("DejaVu Sans", 10, "bold"))\n        self.style.map("TNotebook.Tab",\n                       background=[("selected", self.c_card), ("active", "#2a364d")],\n                       foreground=[("selected", "#ffffff"), ("active", self.c_fg)])\n\n        self.style.configure("Treeview", background="#1a202c", foreground=self.c_fg,\n                             fieldbackground="#1a202c", rowheight=32, font=("DejaVu Sans", 10),\n                             borderwidth=0)\n        self.style.configure("Treeview.Heading", background=self.c_panel, foreground="#ffffff",\n                             font=("DejaVu Sans", 10, "bold"), padding=[6, 6])\n        self.style.map("Treeview", background=[("selected", "#2b4a70")], foreground=[("selected", "#ffffff")])\n\n        self.style.configure("TFrame", background=self.c_bg)\n        self.style.configure("Card.TFrame", background=self.c_panel, relief="flat")\n\n    def create_header(self):\n        header = tk.Frame(self, bg=self.c_panel, height=65)\n        header.pack(fill="x", side="top", padx=12, pady=(10, 6))\n\n        # Brand badge\n        logo_frame = tk.Frame(header, bg=self.c_panel)\n        logo_frame.pack(side="left", padx=14, pady=10)\n\n        lbl_badge = tk.Label(logo_frame, text=" LEGENDARY ", bg=self.c_accent, fg="#ffffff",\n                             font=("DejaVu Sans", 10, "bold"), padx=6, pady=2)\n        lbl_badge.pack(side="left", padx=(0, 10))\n\n        lbl_title = tk.Label(logo_frame, text="Epic Games Launcher", bg=self.c_panel, fg=self.c_fg,\n                             font=("DejaVu Sans", 14, "bold"))\n        lbl_title.pack(side="left")\n\n        # Account & Runner status\n        info_frame = tk.Frame(header, bg=self.c_panel)\n        info_frame.pack(side="right", padx=14, pady=10)\n\n        self.lbl_account = tk.Label(info_frame, text="Checking login status...", bg=self.c_panel,\n                                    fg=self.c_yellow, font=("DejaVu Sans", 10, "bold"))\n        self.lbl_account.pack(anchor="e")\n\n        self.lbl_runner = tk.Label(info_frame, text="Detecting Wine runner...", bg=self.c_panel,\n                                   fg=self.c_muted, font=("DejaVu Sans", 9))\n        self.lbl_runner.pack(anchor="e")\n\n    def create_tabs(self):\n        self.notebook = ttk.Notebook(self)\n        self.notebook.pack(fill="both", expand=True, padx=12, pady=6)\n\n        # Tab 1: Installed Games\n        self.tab_installed = ttk.Frame(self.notebook)\n        self.notebook.add(self.tab_installed, text=" 🎮 Installed Games ")\n        self.build_installed_tab()\n\n        # Tab 2: Epic Library\n        self.tab_library = ttk.Frame(self.notebook)\n        self.notebook.add(self.tab_library, text=" 📥 Full Epic Library ")\n        self.build_library_tab()\n\n        # Tab 3: Console Logs\n        self.tab_console = ttk.Frame(self.notebook)\n        self.notebook.add(self.tab_console, text=" 📜 Launch Console & Logs ")\n        self.build_console_tab()\n\n    def build_installed_tab(self):\n        container = tk.Frame(self.tab_installed, bg=self.c_bg)\n        container.pack(fill="both", expand=True, padx=6, pady=6)\n\n        # Toolbar above games table\n        toolbar = tk.Frame(container, bg=self.c_bg)\n        toolbar.pack(fill="x", side="top", pady=(0, 8))\n\n        btn_launch = tk.Button(toolbar, text=" ▶  LAUNCH SELECTED GAME ", bg=self.c_green, fg="#ffffff",\n                               font=("DejaVu Sans", 11, "bold"), relief="flat", padx=16, pady=6,\n                               activebackground="#27ae60", activeforeground="#ffffff", cursor="hand2",\n                               command=self.launch_selected)\n        btn_launch.pack(side="left", padx=(0, 10))\n\n        btn_sync = tk.Button(toolbar, text=" 🔄 Sync Heroic & Refresh ", bg=self.c_card, fg=self.c_fg,\n                             font=("DejaVu Sans", 10), relief="flat", padx=12, pady=6,\n                             activebackground="#2a364d", cursor="hand2", command=self.sync_and_refresh)\n        btn_sync.pack(side="left", padx=5)\n\n        btn_fix = tk.Button(toolbar, text=" 🛠️ Disarm DXVK & Fix Prefixes ", bg=self.c_card, fg=self.c_fg,\n                            font=("DejaVu Sans", 10), relief="flat", padx=12, pady=6,\n                            activebackground="#2a364d", cursor="hand2", command=self.run_sanitizer)\n        btn_fix.pack(side="left", padx=5)\n\n        btn_login = tk.Button(toolbar, text=" 🔑 Login to Epic Games ", bg=self.c_card, fg="#ffffff",\n                              font=("DejaVu Sans", 10), relief="flat", padx=12, pady=6,\n                              activebackground="#2a364d", cursor="hand2", command=self.open_login_dialog)\n        btn_login.pack(side="right")\n\n        # Table of installed games\n        columns = ("title", "app_name", "version", "size", "install_path")\n        self.tree_installed = ttk.Treeview(container, columns=columns, show="headings", selectmode="browse")\n        self.tree_installed.heading("title", text="Game Title")\n        self.tree_installed.heading("app_name", text="App Name ID")\n        self.tree_installed.heading("version", text="Version")\n        self.tree_installed.heading("size", text="Size")\n        self.tree_installed.heading("install_path", text="Installed Location")\n\n        self.tree_installed.column("title", width=220, anchor="w")\n        self.tree_installed.column("app_name", width=240, anchor="w")\n        self.tree_installed.column("version", width=120, anchor="center")\n        self.tree_installed.column("size", width=90, anchor="center")\n        self.tree_installed.column("install_path", width=250, anchor="w")\n\n        scroll = ttk.Scrollbar(container, orient="vertical", command=self.tree_installed.yview)\n        self.tree_installed.configure(yscrollcommand=scroll.set)\n\n        self.tree_installed.pack(side="left", fill="both", expand=True)\n        scroll.pack(side="right", fill="y")\n\n        self.tree_installed.bind("<Double-1>", lambda e: self.launch_selected())\n\n    def build_library_tab(self):\n        container = tk.Frame(self.tab_library, bg=self.c_bg)\n        container.pack(fill="both", expand=True, padx=6, pady=6)\n\n        toolbar = tk.Frame(container, bg=self.c_bg)\n        toolbar.pack(fill="x", side="top", pady=(0, 8))\n\n        btn_load = tk.Button(toolbar, text=" 🔍 Fetch Library from Epic ", bg=self.c_accent, fg="#ffffff",\n                             font=("DejaVu Sans", 10, "bold"), relief="flat", padx=14, pady=6,\n                             cursor="hand2", command=self.fetch_remote_library)\n        btn_load.pack(side="left", padx=(0, 10))\n\n        btn_install = tk.Button(toolbar, text=" 📥 Install Selected Game ", bg=self.c_card, fg=self.c_fg,\n                                font=("DejaVu Sans", 10), relief="flat", padx=12, pady=6,\n                                cursor="hand2", command=self.install_selected)\n        btn_install.pack(side="left")\n\n        # Filter\n        tk.Label(toolbar, text="Filter:", bg=self.c_bg, fg=self.c_muted, font=("DejaVu Sans", 10)).pack(side="left", padx=(20, 5))\n        self.ent_filter = tk.Entry(toolbar, bg="#1a202c", fg="#ffffff", insertbackground="#ffffff", relief="flat", font=("DejaVu Sans", 10))\n        self.ent_filter.pack(side="left", fill="x", expand=True, padx=(0, 10))\n        self.ent_filter.bind("<KeyRelease>", self.filter_library)\n\n        columns = ("title", "app_name", "status")\n        self.tree_library = ttk.Treeview(container, columns=columns, show="headings", selectmode="browse")\n        self.tree_library.heading("title", text="Game Title")\n        self.tree_library.heading("app_name", text="App Name ID")\n        self.tree_library.heading("status", text="Local Status")\n\n        self.tree_library.column("title", width=400, anchor="w")\n        self.tree_library.column("app_name", width=300, anchor="w")\n        self.tree_library.column("status", width=140, anchor="center")\n\n        scroll = ttk.Scrollbar(container, orient="vertical", command=self.tree_library.yview)\n        self.tree_library.configure(yscrollcommand=scroll.set)\n\n        self.tree_library.pack(side="left", fill="both", expand=True)\n        scroll.pack(side="right", fill="y")\n\n    def build_console_tab(self):\n        container = tk.Frame(self.tab_console, bg=self.c_bg)\n        container.pack(fill="both", expand=True, padx=6, pady=6)\n\n        toolbar = tk.Frame(container, bg=self.c_bg)\n        toolbar.pack(fill="x", side="top", pady=(0, 6))\n\n        btn_stop = tk.Button(toolbar, text=" ⏹  STOP RUNNING GAME ", bg=self.c_red, fg="#ffffff",\n                             font=("DejaVu Sans", 10, "bold"), relief="flat", padx=14, pady=5,\n                             cursor="hand2", command=self.stop_active_game)\n        btn_stop.pack(side="left", padx=(0, 10))\n\n        btn_clear = tk.Button(toolbar, text=" 🧹 Clear Log ", bg=self.c_card, fg=self.c_fg,\n                              font=("DejaVu Sans", 9), relief="flat", padx=10, pady=5,\n                              cursor="hand2", command=lambda: self.log_text.delete("1.0", tk.END))\n        btn_clear.pack(side="left", padx=5)\n\n        self.lbl_status = tk.Label(toolbar, text="Idle", bg=self.c_bg, fg=self.c_muted, font=("DejaVu Sans", 10))\n        self.lbl_status.pack(side="right", padx=10)\n\n        self.log_text = scrolledtext.ScrolledText(container, bg="#0d1117", fg="#c9d1d9",\n                                                 insertbackground="#ffffff", font=("DejaVu Sans Mono", 9),\n                                                 relief="flat", borderwidth=0)\n        self.log_text.pack(fill="both", expand=True)\n\n    def create_bottom_bar(self):\n        bar = tk.Frame(self, bg=self.c_panel, height=32)\n        bar.pack(fill="x", side="bottom", padx=12, pady=(0, 8))\n\n        lbl_tip = tk.Label(bar, text="💡 Tip: Double-click any installed game to launch immediately with WineD3D software rendering.",\n                           bg=self.c_panel, fg=self.c_muted, font=("DejaVu Sans", 9))\n        lbl_tip.pack(side="left", padx=12, pady=6)\n\n    def log(self, text):\n        def _append():\n            self.log_text.insert(tk.END, text + "\\n")\n            self.log_text.see(tk.END)\n        self.after(0, _append)\n\n    def refresh_all(self):\n        self.refresh_account_status()\n        self.refresh_runner_status()\n        self.refresh_installed_games()\n\n    def refresh_account_status(self):\n        user_file = None\n        for p in [HEROIC_LEG / \'user.json\', STD_LEG / \'user.json\']:\n            if p.is_file():\n                user_file = p\n                break\n\n        if user_file:\n            try:\n                data = json.loads(user_file.read_text(encoding=\'utf-8\'))\n                display_name = data.get(\'displayName\', \'Epic User\')\n                acc_id = data.get(\'account_id\', \'\')\n                short_id = f" ({acc_id[:8]}...)" if acc_id else ""\n                self.lbl_account.config(text=f"🟢 Logged In: {display_name}{short_id}", fg=self.c_green)\n                return\n            except Exception:\n                pass\n        self.lbl_account.config(text="🔴 Not Logged In", fg=self.c_red)\n\n    def refresh_runner_status(self):\n        runner = find_best_runner()\n        name = "Wine-GE-Proton8-26" if "Wine-GE" in runner else Path(runner).name\n        self.lbl_runner.config(text=f"🍷 Runner: {name} (WineD3D OpenGL active)")\n\n    def refresh_installed_games(self):\n        for item in self.tree_installed.get_children():\n            self.tree_installed.delete(item)\n\n        installed_file = None\n        for p in [HEROIC_LEG / \'installed.json\', STD_LEG / \'installed.json\']:\n            if p.is_file():\n                installed_file = p\n                break\n\n        games = {}\n        if installed_file:\n            try:\n                games = json.loads(installed_file.read_text(encoding=\'utf-8\'))\n            except Exception as e:\n                self.log(f"Notice reading installed.json: {e}")\n\n        if not games:\n            # Fallback: Query legendary list-installed --json\n            try:\n                res = subprocess.run(["legendary", "list-installed", "--json"], capture_output=True, text=True, timeout=5)\n                if res.returncode == 0 and res.stdout.strip():\n                    data = json.loads(res.stdout)\n                    if isinstance(data, list):\n                        for g in data:\n                            games[g.get(\'app_name\', \'\')] = g\n                    elif isinstance(data, dict):\n                        games = data\n            except Exception:\n                pass\n\n        for app_name, g in sorted(games.items(), key=lambda x: str(x[1].get(\'title\', \'\')).lower()):\n            title = g.get(\'title\') or app_name\n            version = g.get(\'version\', \'Unknown\')\n            size_str = format_size(g.get(\'install_size\', 0))\n            path_str = g.get(\'install_path\', \'N/A\')\n            self.tree_installed.insert("", "end", values=(title, app_name, version, size_str, path_str))\n\n        # Select first item if available\n        children = self.tree_installed.get_children()\n        if children:\n            self.tree_installed.selection_set(children[0])\n            self.tree_installed.focus(children[0])\n\n    def sync_and_refresh(self):\n        self.log("▶ Syncing Heroic and Legendary configurations...")\n        try:\n            fix_script = BENCH / \'fix_heroic_config.py\'\n            if fix_script.is_file():\n                res = subprocess.run([sys.executable, str(fix_script)], capture_output=True, text=True, timeout=10)\n                self.log(res.stdout)\n                if res.stderr:\n                    self.log(res.stderr)\n            self.refresh_all()\n            self.log("✓ Sync completed successfully.")\n        except Exception as exc:\n            self.log(f"Sync error: {exc}")\n\n    def run_sanitizer(self):\n        self.notebook.select(self.tab_console)\n        self.log("▶ Disarming DXVK DLLs and enforcing WineD3D software rendering...")\n        threading.Thread(target=self._run_sanitizer_worker, daemon=True).start()\n\n    def _run_sanitizer_worker(self):\n        try:\n            fix_script = BENCH / \'fix_heroic_config.py\'\n            if fix_script.is_file():\n                res = subprocess.run([sys.executable, str(fix_script)], capture_output=True, text=True, timeout=15)\n                self.log(res.stdout)\n            subprocess.run(["chown", "-R", "gamer:gamer", "/home/gamer"], check=False)\n            subprocess.run(["chmod", "-R", "u+rwX", "/home/gamer"], check=False)\n            self.log("✓ Settings sanitized! DXVK disabled, Wine-GE assigned, and gamer permissions fixed.")\n            self.after(0, self.refresh_all)\n        except Exception as e:\n            self.log(f"Error during sanitization: {e}")\n\n    def launch_selected(self):\n        selected = self.tree_installed.selection()\n        if not selected:\n            messagebox.showinfo("No Selection", "Please select a game from the list to launch.")\n            return\n\n        values = self.tree_installed.item(selected[0], "values")\n        title = values[0]\n        app_name = values[1]\n\n        self.notebook.select(self.tab_console)\n        self.log(f"\\n==================================================================")\n        self.log(f"🚀 Launching game: {title} ({app_name})")\n        self.log(f"==================================================================")\n\n        runner = find_best_runner()\n        self.log(f"Runner: {runner}")\n        self.log(f"WineD3D: Overriding d3d11, dxgi, d3d9, d3d10core = builtin")\n\n        threading.Thread(target=self._launch_worker, args=(app_name, title, runner), daemon=True).start()\n\n    def _launch_worker(self, app_name, title, runner):\n        env = dict(os.environ)\n        env["WINEDLLOVERRIDES"] = "d3d11,dxgi,d3d9,d3d10core=b;mscoree,mshtml=d"\n        env["DISPLAY"] = os.environ.get("DISPLAY", ":0")\n        env["SDL_AUDIODRIVER"] = "pulse"\n        env["HOME"] = "/home/gamer"\n        env["WINEPREFIX"] = str(HOME / ".wine")\n\n        # Complete 32-bit and 64-bit OpenGL & Mesa fallback\n        env["LD_LIBRARY_PATH"] = f"/usr/lib/i386-linux-gnu:/usr/lib/x86_64-linux-gnu:{env.get(\'LD_LIBRARY_PATH\', \'\')}"\n        env["LIBGL_DRIVERS_PATH"] = "/usr/lib/i386-linux-gnu/dri:/usr/lib/x86_64-linux-gnu/dri"\n        env["LIBGL_ALWAYS_SOFTWARE"] = "1"\n        env["MESA_LOADER_DRIVER_OVERRIDE"] = "llvmpipe"\n\n        # Automatically verify and heal Wine prefix before launch if kernel32.dll is missing\n        kernel32 = HOME / ".wine/drive_c/windows/system32/kernel32.dll"\n        if not kernel32.is_file():\n            self.log("▶ Detected incomplete or uninitialized Wine prefix. Auto-initializing with Wine-GE...")\n            wineboot_bin = Path(runner).with_name("wineboot")\n            if wineboot_bin.is_file():\n                try:\n                    init_env = dict(env)\n                    init_env["WINEDLLOVERRIDES"] = "mscoree,mshtml=d"\n                    subprocess.run([str(wineboot_bin), "--init"], env=init_env, check=False, timeout=30)\n                    self.log("✓ Wine prefix auto-initialized successfully.")\n                except Exception as wb_err:\n                    self.log(f"Notice auto-initializing prefix: {wb_err}")\n\n        # Source auto-optimizer Wine settings if available\n        wine_env = HOME / \'.config/gamelab_wine_env.sh\'\n        if wine_env.is_file():\n            for line in wine_env.read_text(encoding=\'utf-8\').splitlines():\n                line = line.strip()\n                if line.startswith(\'export \') and \'=\' in line:\n                    kv = line[7:]  # strip \'export \'\n                    k, _, v = kv.partition(\'=\')\n                    env[k] = v.strip(\'"\').strip("\'")\n\n        cmd = ["legendary", "launch", app_name, "--wine", runner]\n        self.lbl_status.config(text=f"▶ Running: {title}", fg=self.c_green)\n\n        try:\n            self.active_proc = subprocess.Popen(\n                cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env\n            )\n            for line in self.active_proc.stdout:\n                self.log(line.rstrip())\n            self.active_proc.wait()\n            self.log(f"\\nGame process exited with status: {self.active_proc.returncode}")\n        except Exception as exc:\n            self.log(f"Launch exception: {exc}")\n        finally:\n            self.active_proc = None\n            self.lbl_status.config(text="Idle", fg=self.c_muted)\n\n    def stop_active_game(self):\n        if self.active_proc and self.active_proc.poll() is None:\n            self.log("Terminating active game process...")\n            self.active_proc.terminate()\n            time.sleep(1)\n            if self.active_proc.poll() is None:\n                self.active_proc.kill()\n            self.log("Process stopped.")\n        else:\n            self.log("No active game process running.")\n\n    def fetch_remote_library(self):\n        self.log("▶ Fetching Epic Games library via legendary list...")\n        self.ent_filter.delete(0, tk.END)\n        for item in self.tree_library.get_children():\n            self.tree_library.delete(item)\n\n        threading.Thread(target=self._fetch_library_worker, daemon=True).start()\n\n    def _fetch_library_worker(self):\n        try:\n            res = subprocess.run(["legendary", "list", "--json"], capture_output=True, text=True, timeout=25)\n            if res.returncode == 0 and res.stdout.strip():\n                items = json.loads(res.stdout)\n                installed_ids = {self.tree_installed.item(i, "values")[1] for i in self.tree_installed.get_children()}\n\n                self.all_library_items = []\n                for item in items:\n                    app = item.get("app_name", "")\n                    title = item.get("app_title") or item.get("title") or app\n                    status = "✓ Installed" if app in installed_ids else "Available"\n                    self.all_library_items.append((title, app, status))\n\n                self.all_library_items.sort(key=lambda x: x[0].lower())\n                self.after(0, self._populate_library_table, self.all_library_items)\n                self.log(f"✓ Found {len(self.all_library_items)} games in your Epic library.")\n            else:\n                self.log(f"Legendary list returned code {res.returncode}:\\n{res.stderr or res.stdout}")\n        except Exception as e:\n            self.log(f"Library fetch error: {e}")\n\n    def _populate_library_table(self, items):\n        for item in self.tree_library.get_children():\n            self.tree_library.delete(item)\n        for row in items:\n            self.tree_library.insert("", "end", values=row)\n\n    def filter_library(self, event=None):\n        query = self.ent_filter.get().strip().lower()\n        if not hasattr(self, \'all_library_items\'):\n            return\n        if not query:\n            self._populate_library_table(self.all_library_items)\n        else:\n            filtered = [r for r in self.all_library_items if query in r[0].lower() or query in r[1].lower()]\n            self._populate_library_table(filtered)\n\n    def install_selected(self):\n        sel = self.tree_library.selection()\n        if not sel:\n            messagebox.showinfo("No Selection", "Please select a game to install from the library.")\n            return\n\n        values = self.tree_library.item(sel[0], "values")\n        title = values[0]\n        app_name = values[1]\n\n        if not messagebox.askyesno("Install Game", f"Install \'{title}\' ({app_name}) via Legendary?"):\n            return\n\n        self.notebook.select(self.tab_console)\n        self.log(f"\\n▶ Starting installation of {title} ({app_name})...")\n        threading.Thread(target=self._install_worker, args=(app_name,), daemon=True).start()\n\n    def _install_worker(self, app_name):\n        try:\n            cmd = ["legendary", "install", app_name, "-y"]\n            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)\n            for line in proc.stdout:\n                self.log(line.rstrip())\n            proc.wait()\n            self.log(f"Installation finished with code {proc.returncode}.")\n            self.sync_and_refresh()\n        except Exception as e:\n            self.log(f"Install error: {e}")\n\n    def open_login_dialog(self):\n        dlg = tk.Toplevel(self)\n        dlg.title("Epic Games Authentication")\n        dlg.geometry("560x380")\n        dlg.resizable(False, False)\n        dlg.configure(bg=self.c_bg)\n        dlg.transient(self)\n        dlg.grab_set()\n\n        tk.Label(dlg, text="Log In to Epic Games", bg=self.c_bg, fg="#ffffff",\n                 font=("DejaVu Sans", 13, "bold")).pack(pady=(16, 10))\n\n        frame_steps = tk.Frame(dlg, bg=self.c_panel, padx=14, pady=12)\n        frame_steps.pack(fill="x", padx=16, pady=6)\n\n        msg = (\n            "1. Open this URL in your PC browser (Chrome/Edge/Firefox):\\n"\n            "   https://legendary.gl/epiclogin\\n\\n"\n            "2. Sign in with your Epic account.\\n\\n"\n            "3. The browser will show text containing \'authorizationCode\'.\\n"\n            "4. Copy and paste the code (or the entire JSON string) below:"\n        )\n        tk.Label(frame_steps, text=msg, bg=self.c_panel, fg=self.c_fg, justify="left",\n                 font=("DejaVu Sans", 9)).pack(anchor="w")\n\n        # Copy URL button\n        def _copy_url():\n            self.clipboard_clear()\n            self.clipboard_append("https://legendary.gl/epiclogin")\n            lbl_info.config(text="✓ Link copied to clipboard!", fg=self.c_green)\n\n        btn_copy = tk.Button(frame_steps, text=" 📋 Copy Login Link ", bg=self.c_card, fg=self.c_fg,\n                             font=("DejaVu Sans", 9), relief="flat", cursor="hand2", command=_copy_url)\n        btn_copy.pack(anchor="w", pady=(6, 0))\n\n        # Input entry\n        lbl_code = tk.Label(dlg, text="Authorization Code or JSON Text:", bg=self.c_bg, fg=self.c_muted,\n                            font=("DejaVu Sans", 9, "bold"))\n        lbl_code.pack(anchor="w", padx=18, pady=(10, 2))\n\n        ent_code = tk.Entry(dlg, bg="#1a202c", fg="#ffffff", insertbackground="#ffffff",\n                            relief="flat", font=("DejaVu Sans Mono", 10))\n        ent_code.pack(fill="x", padx=18, ipady=4)\n\n        lbl_info = tk.Label(dlg, text="", bg=self.c_bg, fg=self.c_yellow, font=("DejaVu Sans", 9))\n        lbl_info.pack(pady=4)\n\n        def _submit():\n            raw_input = ent_code.get().strip()\n            if not raw_input:\n                lbl_info.config(text="Please paste an authorization code first.", fg=self.c_red)\n                return\n\n            # Extract code if full JSON was pasted\n            code = raw_input\n            if "{" in raw_input and "authorizationCode" in raw_input:\n                try:\n                    data = json.loads(raw_input)\n                    code = data.get("authorizationCode", raw_input)\n                except Exception:\n                    m = re.search(r\'"authorizationCode"\\s*:\\s*"([a-f0-9]+)"\', raw_input, re.I)\n                    if m:\n                        code = m.group(1)\n\n            lbl_info.config(text="Authenticating with Epic Games...", fg=self.c_yellow)\n            dlg.update()\n\n            def _auth_worker():\n                res = subprocess.run(["legendary", "auth", "--code", code], capture_output=True, text=True)\n                if res.returncode == 0:\n                    self.sync_and_refresh()\n                    self.after(0, lambda: (\n                        messagebox.showinfo("Success", "Epic Games authentication succeeded!"),\n                        dlg.destroy()\n                    ))\n                else:\n                    err_msg = res.stderr or res.stdout\n                    self.after(0, lambda: lbl_info.config(text=f"Auth failed: {err_msg[:60]}", fg=self.c_red))\n\n            threading.Thread(target=_auth_worker, daemon=True).start()\n\n        btn_frame = tk.Frame(dlg, bg=self.c_bg)\n        btn_frame.pack(fill="x", padx=18, pady=10)\n\n        tk.Button(btn_frame, text=" Cancel ", bg=self.c_card, fg=self.c_muted, relief="flat",\n                  cursor="hand2", command=dlg.destroy).pack(side="left")\n\n        tk.Button(btn_frame, text=" 🚀 Log In to Epic Games ", bg=self.c_green, fg="#ffffff",\n                  font=("DejaVu Sans", 10, "bold"), relief="flat", padx=14, pady=5,\n                  cursor="hand2", command=_submit).pack(side="right")\n\nif __name__ == "__main__":\n    app = LegendaryApp()\n    app.mainloop()\n', encoding='utf-8')

legendary_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export HOME="/home/gamer"
mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/legendary_gui.log"

for w in /home/gamer/.config/heroic/tools/wine/*/bin/wine; do
    if [ -f "$w" ]; then
        if [ ! -f /usr/bin/wine ]; then
            ln -sf "$w" /usr/bin/wine 2>/dev/null || true
            ln -sf "$(dirname "$w")/wineserver" /usr/bin/wineserver 2>/dev/null || true
            ln -sf "$(dirname "$w")/wineboot" /usr/bin/wineboot 2>/dev/null || true
        fi
        if [ ! -f /usr/local/bin/wine ]; then
            ln -sf "$w" /usr/local/bin/wine 2>/dev/null || true
        fi
        break
    fi
done

if python3 -c "import tkinter" >/dev/null 2>&1; then
    exec python3 /tmp/kaggle-game-dev-bench/legendary_gui.py >> "$LOG" 2>&1
else
    if [ "$(id -u)" -eq 0 ]; then
        apt-get update -qq && apt-get install -y -qq python3-tk >> "$LOG" 2>&1 || true
        if python3 -c "import tkinter" >/dev/null 2>&1; then
            exec python3 /tmp/kaggle-game-dev-bench/legendary_gui.py >> "$LOG" 2>&1
        fi
    fi
    exec xterm -geometry 105x32 -title "Legendary Epic Games Manager" -e /bin/bash /tmp/kaggle-game-dev-bench/legendary_menu.sh >> "$LOG" 2>&1
fi
"""
(BENCH / 'launch_legendary.sh').write_text(legendary_sh, encoding='utf-8')

# Auto-Optimizer script and launch wrapper
(BENCH / 'auto_optimize.py').write_text('"""Auto-Optimize: One-click adaptive quality tuner for Game Lab streaming & gaming."""\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport threading\nimport time\nimport tkinter as tk\nfrom tkinter import ttk, messagebox\nimport urllib.request\n\nHOME = Path(os.environ.get(\'KAGGLE_GAMER_HOME\', \'/home/gamer\'))\nBENCH = Path(\'/tmp/kaggle-game-dev-bench\')\nWINE_ENV_FILE = HOME / \'.config/gamelab_wine_env.sh\'\nSETTINGS_FILE = HOME / \'.config/gamelab_optimize.json\'\n\n# ---------------------------------------------------------------------------\n# Quality presets\n# ---------------------------------------------------------------------------\nPRESETS = {\n    \'low\': {\n        \'label\': \'📶  Low  (بطيء)\',\n        \'resolution\': \'854x480\',\n        \'wine_csmt\': False,\n        \'wine_extra\': \'renderer=gl\',\n        \'desc\': \'Best for slow or mobile connections (< 5 Mbps).\\nLower resolution but very smooth streaming.\',\n    },\n    \'medium\': {\n        \'label\': \'📶📶  Medium  (متوسط)\',\n        \'resolution\': \'1024x576\',\n        \'wine_csmt\': False,\n        \'wine_extra\': \'renderer=gl\',\n        \'desc\': \'Balanced quality for average connections (5–15 Mbps).\\nGood balance between clarity and smoothness.\',\n    },\n    \'high\': {\n        \'label\': \'📶📶📶  High  (عالي)\',\n        \'resolution\': \'1280x720\',\n        \'wine_csmt\': False,\n        \'wine_extra\': \'\',\n        \'desc\': \'Full 720p HD at 60 FPS (15–30 Mbps).\\nRecommended for broadband connections.\',\n    },\n    \'ultra\': {\n        \'label\': \'📶📶📶📶  Ultra  (أعلى جودة)\',\n        \'resolution\': \'1280x720\',\n        \'wine_csmt\': True,\n        \'wine_extra\': \'\',\n        \'desc\': \'Maximum quality with all renderer features (30+ Mbps).\\nRequires fast, stable connection.\',\n    },\n}\nPRESET_ORDER = [\'low\', \'medium\', \'high\', \'ultra\']\n\n# ---------------------------------------------------------------------------\n# Keyboard Lock JavaScript (injected into Selkies HTML)\n# ---------------------------------------------------------------------------\nKEYBOARD_LOCK_JS = """\n<!-- Game Lab: Keyboard Lock – prevents Esc from exiting browser fullscreen -->\n<script>\n(function(){\n  var _kl=false;\n  function _lock(){\n    if(document.fullscreenElement && navigator.keyboard && navigator.keyboard.lock){\n      navigator.keyboard.lock([\'Escape\']).then(function(){_kl=true;\n        console.log(\'[GameLab] Keyboard locked – Esc captured for gaming\');\n      }).catch(function(e){console.log(\'[GameLab] Keyboard lock:\',e.message);});\n    }\n  }\n  document.addEventListener(\'fullscreenchange\',function(){\n    if(document.fullscreenElement){_lock();}\n    else if(_kl && navigator.keyboard && navigator.keyboard.unlock){navigator.keyboard.unlock();_kl=false;}\n  });\n  _lock();\n})();\n</script>\n"""\n\n# ---------------------------------------------------------------------------\n# Helpers\n# ---------------------------------------------------------------------------\n\ndef get_current_resolution():\n    try:\n        r = subprocess.run([\'xrandr\', \'--query\'], capture_output=True, text=True, timeout=5)\n        for line in r.stdout.splitlines():\n            if \'*\' in line:\n                return line.strip().split()[0]\n    except Exception:\n        pass\n    return \'1280x720\'\n\n\ndef set_resolution(res):\n    try:\n        w, h = res.split(\'x\')\n        # Try to set directly first\n        r = subprocess.run([\'xrandr\', \'-s\', res], capture_output=True, text=True, timeout=5)\n        if r.returncode != 0:\n            # Create the mode and try again\n            cvt = subprocess.run([\'cvt\', w, h], capture_output=True, text=True, timeout=5)\n            for line in cvt.stdout.splitlines():\n                if \'Modeline\' in line:\n                    parts = line.split(\'"\')\n                    if len(parts) >= 3:\n                        mode_name = parts[1]\n                        mode_args = parts[2].strip()\n                        subprocess.run([\'xrandr\', \'--newmode\', mode_name] + mode_args.split(), check=False, timeout=5)\n                        for output in (\'screen\', \'SCREEN\', \'default\', \'VGA-0\', \'Virtual-1\'):\n                            subprocess.run([\'xrandr\', \'--addmode\', output, mode_name], check=False, timeout=5)\n                        subprocess.run([\'xrandr\', \'-s\', mode_name], check=False, timeout=5)\n        return True\n    except Exception as e:\n        print(f\'Resolution change error: {e}\')\n        return False\n\n\ndef write_wine_env(preset_key):\n    p = PRESETS[preset_key]\n    csmt = \'1\' if p[\'wine_csmt\'] else \'0\'\n    d3d = f"csmt={csmt}"\n    if p.get(\'wine_extra\'):\n        d3d += \';\' + p[\'wine_extra\']\n\n    content = f"""#!/bin/bash\n# Auto-generated by Game Lab Auto-Optimizer – preset: {preset_key}\nexport WINE_D3D_CONFIG="{d3d}"\nexport MESA_GL_VERSION_OVERRIDE="4.3"\nexport __GL_SYNC_TO_VBLANK=0\nexport vblank_mode=0\nexport WINEDLLOVERRIDES="d3d11,dxgi,d3d9,d3d10core=b"\n"""\n    WINE_ENV_FILE.parent.mkdir(parents=True, exist_ok=True)\n    WINE_ENV_FILE.write_text(content, encoding=\'utf-8\')\n\n\ndef save_settings(preset_key):\n    SETTINGS_FILE.parent.mkdir(parents=True, exist_ok=True)\n    SETTINGS_FILE.write_text(json.dumps({\'preset\': preset_key, \'ts\': time.time()}, indent=2), encoding=\'utf-8\')\n\n\ndef load_settings():\n    try:\n        return json.loads(SETTINGS_FILE.read_text(encoding=\'utf-8\')).get(\'preset\', \'high\')\n    except Exception:\n        return \'high\'\n\n\ndef inject_keyboard_lock():\n    """Inject keyboard lock JS into every Selkies HTML file found on disk."""\n    marker = \'Game Lab: Keyboard Lock\'\n    patched = []\n    search_roots = [Path(\'/usr/local/lib\'), Path(\'/usr/lib\')]\n    for root in search_roots:\n        if not root.is_dir():\n            continue\n        for html in root.rglob(\'index.html\'):\n            if \'selkies\' not in str(html).lower():\n                continue\n            try:\n                text = html.read_text(encoding=\'utf-8\')\n                if marker in text:\n                    patched.append(f\'{html} (already patched)\')\n                    continue\n                if \'</head>\' in text:\n                    html.write_text(text.replace(\'</head>\', KEYBOARD_LOCK_JS + \'\\n</head>\'), encoding=\'utf-8\')\n                    patched.append(str(html))\n                elif \'</body>\' in text:\n                    html.write_text(text.replace(\'</body>\', KEYBOARD_LOCK_JS + \'\\n</body>\'), encoding=\'utf-8\')\n                    patched.append(str(html))\n            except Exception as e:\n                patched.append(f\'{html} (error: {e})\')\n    return patched\n\n\ndef test_bandwidth():\n    """Quick bandwidth measurement (server-side egress estimate)."""\n    urls = [\n        \'https://speed.cloudflare.com/__down?bytes=2000000\',\n        \'https://www.google.com/images/branding/googlelogo/2x/googlelogo_color_272x92dp.png\',\n    ]\n    for url in urls:\n        try:\n            req = urllib.request.Request(url, headers={\'User-Agent\': \'GameLab/1.0\'})\n            t0 = time.time()\n            with urllib.request.urlopen(req, timeout=8) as resp:\n                data = resp.read()\n            elapsed = time.time() - t0\n            if elapsed > 0.01:\n                return (len(data) * 8) / (elapsed * 1_000_000)\n        except Exception:\n            continue\n    return 0.0\n\n\ndef recommend_preset(speed_mbps):\n    if speed_mbps >= 30:\n        return \'ultra\'\n    elif speed_mbps >= 15:\n        return \'high\'\n    elif speed_mbps >= 5:\n        return \'medium\'\n    return \'low\'\n\n\n# ---------------------------------------------------------------------------\n# GUI\n# ---------------------------------------------------------------------------\n\nclass AutoOptimizer(tk.Tk):\n    def __init__(self):\n        super().__init__()\n        self.title("⚡ Game Lab Auto-Optimizer  (تلقائي)")\n        self.geometry("720x560")\n        self.minsize(640, 480)\n\n        # Dark theme\n        self.c_bg = "#141923"\n        self.c_panel = "#1b2230"\n        self.c_card = "#222c3e"\n        self.c_fg = "#f0f4fc"\n        self.c_muted = "#8fa0bd"\n        self.c_accent = "#3498db"\n        self.c_green = "#2ecc71"\n        self.c_red = "#e74c3c"\n        self.c_yellow = "#f39c12"\n        self.c_orange = "#e67e22"\n\n        self.configure(bg=self.c_bg)\n        self.current_preset = load_settings()\n\n        self.build_header()\n        self.build_status_panel()\n        self.build_presets_panel()\n        self.build_actions_panel()\n        self.build_footer()\n\n        self.refresh_status()\n\n    # ---- Header ----\n    def build_header(self):\n        hdr = tk.Frame(self, bg=self.c_panel)\n        hdr.pack(fill=\'x\', padx=10, pady=(8, 4))\n\n        tk.Label(hdr, text=" ⚡ AUTO-OPTIMIZE ", bg=self.c_orange, fg=\'#fff\',\n                 font=(\'DejaVu Sans\', 10, \'bold\'), padx=6, pady=2).pack(side=\'left\', padx=(10, 12), pady=8)\n        tk.Label(hdr, text="Adaptive Streaming & Rendering Quality", bg=self.c_panel, fg=self.c_fg,\n                 font=(\'DejaVu Sans\', 13, \'bold\')).pack(side=\'left\', pady=8)\n\n    # ---- Status panel ----\n    def build_status_panel(self):\n        frm = tk.Frame(self, bg=self.c_card, padx=14, pady=10)\n        frm.pack(fill=\'x\', padx=10, pady=6)\n\n        tk.Label(frm, text="Current Status", bg=self.c_card, fg=\'#fff\',\n                 font=(\'DejaVu Sans\', 11, \'bold\')).grid(row=0, column=0, columnspan=4, sticky=\'w\', pady=(0, 6))\n\n        self.lbl_resolution = self._stat(frm, 1, 0, "Resolution:")\n        self.lbl_preset = self._stat(frm, 1, 2, "Active Preset:")\n        self.lbl_wine = self._stat(frm, 2, 0, "Wine Renderer:")\n        self.lbl_keyboard = self._stat(frm, 2, 2, "Esc Key Lock:")\n\n    def _stat(self, parent, row, col, label):\n        tk.Label(parent, text=label, bg=self.c_card, fg=self.c_muted,\n                 font=(\'DejaVu Sans\', 9, \'bold\')).grid(row=row, column=col, sticky=\'w\', padx=(0, 6), pady=2)\n        val = tk.Label(parent, text="...", bg=self.c_card, fg=self.c_fg, font=(\'DejaVu Sans\', 9))\n        val.grid(row=row, column=col + 1, sticky=\'w\', padx=(0, 30), pady=2)\n        return val\n\n    # ---- Presets panel ----\n    def build_presets_panel(self):\n        outer = tk.Frame(self, bg=self.c_bg)\n        outer.pack(fill=\'both\', expand=True, padx=10, pady=6)\n\n        tk.Label(outer, text="Select Quality Preset:", bg=self.c_bg, fg=\'#fff\',\n                 font=(\'DejaVu Sans\', 11, \'bold\')).pack(anchor=\'w\', pady=(0, 6))\n\n        self.preset_btns = {}\n        for key in PRESET_ORDER:\n            p = PRESETS[key]\n            card = tk.Frame(outer, bg=self.c_card, padx=12, pady=8, cursor=\'hand2\',\n                            highlightthickness=2, highlightbackground=self.c_card)\n            card.pack(fill=\'x\', pady=3)\n\n            lbl = tk.Label(card, text=p[\'label\'], bg=self.c_card, fg=self.c_fg,\n                           font=(\'DejaVu Sans\', 11, \'bold\'), anchor=\'w\')\n            lbl.pack(side=\'left\', padx=(0, 12))\n\n            desc_short = p[\'desc\'].split(\'\\n\')[0]\n            tk.Label(card, text=desc_short, bg=self.c_card, fg=self.c_muted,\n                     font=(\'DejaVu Sans\', 9), anchor=\'w\').pack(side=\'left\', fill=\'x\', expand=True)\n\n            res_lbl = tk.Label(card, text=p[\'resolution\'], bg=self.c_card, fg=self.c_accent,\n                               font=(\'DejaVu Sans Mono\', 10, \'bold\'))\n            res_lbl.pack(side=\'right\', padx=8)\n\n            self.preset_btns[key] = card\n            for widget in (card, lbl, res_lbl):\n                widget.bind(\'<Button-1>\', lambda e, k=key: self.select_preset(k))\n\n    def select_preset(self, key):\n        self.current_preset = key\n        for k, card in self.preset_btns.items():\n            if k == key:\n                card.configure(highlightbackground=self.c_accent, bg=\'#1a3a52\')\n                for w in card.winfo_children():\n                    w.configure(bg=\'#1a3a52\')\n            else:\n                card.configure(highlightbackground=self.c_card, bg=self.c_card)\n                for w in card.winfo_children():\n                    w.configure(bg=self.c_card)\n\n    # ---- Actions panel ----\n    def build_actions_panel(self):\n        frm = tk.Frame(self, bg=self.c_bg)\n        frm.pack(fill=\'x\', padx=10, pady=8)\n\n        btn_auto = tk.Button(frm, text=" 🚀  AUTO-DETECT & APPLY ",\n                             bg=self.c_green, fg=\'#fff\', font=(\'DejaVu Sans\', 11, \'bold\'),\n                             relief=\'flat\', padx=16, pady=8, cursor=\'hand2\',\n                             activebackground=\'#27ae60\', command=self.auto_detect_and_apply)\n        btn_auto.pack(side=\'left\', padx=(0, 10))\n\n        btn_apply = tk.Button(frm, text=" ✅  Apply Selected Preset ",\n                              bg=self.c_accent, fg=\'#fff\', font=(\'DejaVu Sans\', 10, \'bold\'),\n                              relief=\'flat\', padx=14, pady=8, cursor=\'hand2\',\n                              activebackground=\'#2980b9\', command=self.apply_preset)\n        btn_apply.pack(side=\'left\', padx=5)\n\n        btn_esc = tk.Button(frm, text=" 🔒  Fix Esc Key ",\n                            bg=self.c_card, fg=self.c_fg, font=(\'DejaVu Sans\', 10),\n                            relief=\'flat\', padx=12, pady=8, cursor=\'hand2\',\n                            command=self.fix_esc_key)\n        btn_esc.pack(side=\'left\', padx=5)\n\n        self.lbl_action_status = tk.Label(frm, text="", bg=self.c_bg, fg=self.c_yellow,\n                                          font=(\'DejaVu Sans\', 9))\n        self.lbl_action_status.pack(side=\'right\', padx=10)\n\n    # ---- Footer ----\n    def build_footer(self):\n        bar = tk.Frame(self, bg=self.c_panel, height=28)\n        bar.pack(fill=\'x\', side=\'bottom\', padx=10, pady=(0, 6))\n        tk.Label(bar, text="💡 Tip: Press AUTO-DETECT for the best settings, or pick a preset and click Apply.",\n                 bg=self.c_panel, fg=self.c_muted, font=(\'DejaVu Sans\', 9)).pack(padx=10, pady=5)\n\n    # ---- Logic ----\n\n    def refresh_status(self):\n        cur_res = get_current_resolution()\n        self.lbl_resolution.config(text=cur_res)\n        self.lbl_preset.config(text=PRESETS.get(self.current_preset, {}).get(\'label\', self.current_preset))\n\n        # Wine renderer info\n        env_csmt = \'Enabled\' if PRESETS.get(self.current_preset, {}).get(\'wine_csmt\') else \'Disabled\'\n        self.lbl_wine.config(text=f"WineD3D (OpenGL) · CSMT {env_csmt}")\n\n        # Keyboard lock status\n        kl_status = "Not checked"\n        try:\n            for root in [Path(\'/usr/local/lib\'), Path(\'/usr/lib\')]:\n                for html in root.rglob(\'index.html\'):\n                    if \'selkies\' in str(html).lower():\n                        if \'Game Lab: Keyboard Lock\' in html.read_text(encoding=\'utf-8\'):\n                            kl_status = "✅ Active (Esc captured)"\n                        else:\n                            kl_status = "❌ Not injected"\n                        break\n        except Exception:\n            kl_status = "Unknown"\n        self.lbl_keyboard.config(text=kl_status)\n\n        self.select_preset(self.current_preset)\n\n    def apply_preset(self):\n        self.lbl_action_status.config(text="Applying...", fg=self.c_yellow)\n        self.update()\n        threading.Thread(target=self._apply_worker, daemon=True).start()\n\n    def _apply_worker(self):\n        key = self.current_preset\n        p = PRESETS[key]\n        msgs = []\n\n        # 1. Resolution\n        if set_resolution(p[\'resolution\']):\n            msgs.append(f"Resolution → {p[\'resolution\']}")\n        else:\n            msgs.append(f"Resolution change to {p[\'resolution\']} failed (may already be set)")\n\n        # 2. Wine env\n        write_wine_env(key)\n        msgs.append(f"Wine config → csmt={\'on\' if p[\'wine_csmt\'] else \'off\'}")\n\n        # 3. Save\n        save_settings(key)\n        msgs.append(f"Preset saved: {key}")\n\n        summary = " · ".join(msgs)\n        self.after(0, lambda: self.lbl_action_status.config(text=f"✅ {summary}", fg=self.c_green))\n        self.after(0, self.refresh_status)\n\n    def auto_detect_and_apply(self):\n        self.lbl_action_status.config(text="🔍 Testing bandwidth...", fg=self.c_yellow)\n        self.update()\n        threading.Thread(target=self._auto_worker, daemon=True).start()\n\n    def _auto_worker(self):\n        speed = test_bandwidth()\n        recommended = recommend_preset(speed)\n\n        self.current_preset = recommended\n        self.after(0, lambda: self.select_preset(recommended))\n        self.after(0, lambda: self.lbl_action_status.config(\n            text=f"📶 Speed: {speed:.1f} Mbps → Recommended: {PRESETS[recommended][\'label\']}",\n            fg=self.c_accent))\n        self.after(100, lambda: None)  # small delay\n\n        # Apply\n        p = PRESETS[recommended]\n        set_resolution(p[\'resolution\'])\n        write_wine_env(recommended)\n        save_settings(recommended)\n\n        # Also inject keyboard lock\n        patched = inject_keyboard_lock()\n\n        self.after(200, lambda: self.lbl_action_status.config(\n            text=f"✅ Applied: {PRESETS[recommended][\'label\']} ({p[\'resolution\']}) · {speed:.1f} Mbps" +\n                 (" · Esc Lock ✅" if patched else ""),\n            fg=self.c_green))\n        self.after(200, self.refresh_status)\n\n    def fix_esc_key(self):\n        self.lbl_action_status.config(text="🔒 Injecting keyboard lock...", fg=self.c_yellow)\n        self.update()\n\n        def _worker():\n            patched = inject_keyboard_lock()\n            if patched:\n                msg = f"✅ Keyboard lock injected into {len(patched)} file(s). Reload your browser tab (F5) to activate."\n            else:\n                msg = "⚠️ Could not find Selkies HTML files. Use the browser bookmarklet instead."\n            self.after(0, lambda: self.lbl_action_status.config(text=msg, fg=self.c_green if patched else self.c_yellow))\n            self.after(0, self.refresh_status)\n\n            if not patched:\n                self.after(500, self.show_bookmarklet_fallback)\n\n        threading.Thread(target=_worker, daemon=True).start()\n\n    def show_bookmarklet_fallback(self):\n        dlg = tk.Toplevel(self)\n        dlg.title("🔒 Esc Key Fix – Browser Bookmarklet")\n        dlg.geometry("540x260")\n        dlg.configure(bg=self.c_bg)\n        dlg.transient(self)\n\n        tk.Label(dlg, text="Alternative: Browser Bookmarklet", bg=self.c_bg, fg=\'#fff\',\n                 font=(\'DejaVu Sans\', 12, \'bold\')).pack(pady=(12, 8))\n\n        tk.Label(dlg, text=(\n            "If the automatic injection didn\'t work, you can fix the Esc key\\n"\n            "by creating a browser bookmark with this code.\\n"\n            "Click the bookmark after entering fullscreen (F11)."\n        ), bg=self.c_bg, fg=self.c_muted, font=(\'DejaVu Sans\', 9), justify=\'left\').pack(padx=14)\n\n        code = "javascript:void(navigator.keyboard&&navigator.keyboard.lock([\'Escape\']).then(()=>document.title+=\' 🔒\').catch(e=>alert(\'Keyboard lock: \'+e)))"\n        ent = tk.Entry(dlg, bg=\'#1a202c\', fg=\'#0f0\', insertbackground=\'#fff\',\n                       font=(\'DejaVu Sans Mono\', 8), relief=\'flat\')\n        ent.pack(fill=\'x\', padx=14, pady=8, ipady=4)\n        ent.insert(0, code)\n        ent.select_range(0, tk.END)\n\n        def _copy():\n            self.clipboard_clear()\n            self.clipboard_append(code)\n            btn_copy.config(text="✓ Copied!")\n\n        btn_copy = tk.Button(dlg, text=" 📋 Copy to Clipboard ", bg=self.c_card, fg=self.c_fg,\n                             relief=\'flat\', cursor=\'hand2\', command=_copy)\n        btn_copy.pack(pady=6)\n\n\nif __name__ == "__main__":\n    app = AutoOptimizer()\n    app.mainloop()\n', encoding='utf-8')

optimizer_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export HOME="/home/gamer"
mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/auto_optimize.log"

if ! python3 -c "import tkinter" >/dev/null 2>&1; then
    apt-get update -qq && apt-get install -y -qq python3-tk >> "$LOG" 2>&1 || true
fi

exec python3 /tmp/kaggle-game-dev-bench/auto_optimize.py >> "$LOG" 2>&1
"""
(BENCH / 'launch_optimizer.sh').write_text(optimizer_sh, encoding='utf-8')

# Make all shell scripts executable
for p in BENCH.glob('*.sh'):
    p.chmod(0o755)

print('Launcher scripts, storefront wrappers, and desktop configuration saved successfully.', flush=True)


In [ ]:
from pathlib import Path
import json, os, shutil, signal, subprocess, sys, time, urllib.request

BENCH = Path('/tmp/kaggle-game-dev-bench')
BUILDS = Path('/tmp/game-builds')
BUILDS.mkdir(parents=True, exist_ok=True)
SCREENSHOTS = Path('/kaggle/working/screenshots')
SCREENSHOTS.mkdir(parents=True, exist_ok=True)

# Master desktop session wrapper
wrapper = BENCH / 'start-desktop.sh'
wrapper.write_text('''#!/bin/bash
export LD_PRELOAD='/tmp/kaggle-game-dev-bench/$LIB/selkies_input_interposer.so:/tmp/kaggle-game-dev-bench/$LIB/selkies_fake_udev.so'
export SDL_AUDIODRIVER=pulse
export SELKIES_JS_SOCKET_PATH="$XDG_RUNTIME_DIR"
export SDL_JOYSTICK_DISABLE_UDEV=1
export SDL_JOYSTICK_DEVICE=/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
printf '%s' "$XAUTHORITY" > /tmp/kaggle-game-dev-bench/xauthority-path
printf '%s' "$PULSE_SERVER" > /tmp/kaggle-game-dev-bench/pulse-server-path

# Share the session PulseAudio server with the unprivileged desktop user.
install -d -o gamer -g gamer -m 700 /home/gamer/.config/pulse
if [ -f "$HOME/.config/pulse/cookie" ]; then
    install -o gamer -g gamer -m 600 "$HOME/.config/pulse/cookie" /home/gamer/.config/pulse/cookie
fi

# Grant unprivileged gamer user access to X11 display
echo "X11 authorization: $(xhost +si:localuser:gamer 2>&1)"

# Grant unprivileged gamer user access to PulseAudio socket and runtime directory
chmod 777 "$XDG_RUNTIME_DIR" 2>/dev/null || true
chmod -R 777 "$XDG_RUNTIME_DIR/pulse" 2>/dev/null || true

# Start Openbox window manager in background for window frames, moving & resizing
openbox &

# Start tint2 taskbar panel in background for window switching
if which tint2 >/dev/null 2>&1; then
    tint2 -c /tmp/kaggle-game-dev-bench/tint2rc &
fi

# Run the controller-friendly launcher UI as the main session process
exec runuser -u gamer -- env -u DBUS_SESSION_BUS_ADDRESS DISPLAY="$DISPLAY" HOME=/home/gamer PATH="$PATH" \
    XDG_RUNTIME_DIR="$XDG_RUNTIME_DIR" SDL_AUDIODRIVER=pulse \
    PULSE_SERVER="$PULSE_SERVER" \
    SELKIES_JS_SOCKET_PATH="$SELKIES_JS_SOCKET_PATH" \
    LD_PRELOAD="$LD_PRELOAD" SDL_JOYSTICK_DISABLE_UDEV=1 \
    SDL_JOYSTICK_DEVICE="$SDL_JOYSTICK_DEVICE" \
    dbus-run-session -- /usr/bin/env python3 /tmp/kaggle-game-dev-bench/dev_launcher.py
''')
wrapper.chmod(0o755)

# Stop any previous session
pidfile = BENCH / 'selkies.pid'
if pidfile.exists():
    try:
        previous = int(pidfile.read_text())
        command = Path(f'/proc/{previous}/cmdline').read_bytes().replace(b'\0', b' ')
        if b'selkies-session' in command and b'kaggle-game-dev-bench' in command:
            os.killpg(previous, signal.SIGTERM)
            time.sleep(2)
    except (ValueError, OSError, ProcessLookupError):
        pass

env = os.environ.copy()
env['PATH'] = f"/usr/local/bin:/usr/games:/usr/bin:{env.get('PATH', '')}"
env.update(SELKIES_ENABLE_HTTPS='false', SELKIES_MODE='websockets',
           SELKIES_ALLOWED_ORIGINS='https://kkb-production.jupyter-proxy.kaggle.net',
           SELKIES_ENCODER='h264enc', SELKIES_FRAMERATE='60',
           SELKIES_AUDIO_ENABLED='true', SELKIES_AUDIO_CHANNELS='2',
           SELKIES_AUDIO_BITRATE='128000', SELKIES_GAMEPAD_ENABLED='true',
           SELKIES_UINPUT_GAMEPAD='false',
           SELKIES_UI_TITLE='Linux Gaming & Game-Dev Environment',
           SELKIES_MANUAL_WIDTH='1280', SELKIES_MANUAL_HEIGHT='720',
           FILE_MANAGER_PATH=str(BUILDS),
           SDL_AUDIODRIVER='pulse')

log = (BENCH / 'selkies.log').open('a', buffering=1)
server = subprocess.Popen(['selkies-session', '--session=' + str(wrapper),
                           '--port=8080', '--enable-basic-auth=false'],
                          stdout=log, stderr=subprocess.STDOUT,
                          stdin=subprocess.DEVNULL, env=env,
                          start_new_session=True)
pidfile.write_text(str(server.pid))

health = None
for _ in range(75):
    if server.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8080/api/health', timeout=2) as response:
            if response.status == 200:
                health = response.read().decode(errors='replace')
                break
    except Exception:
        time.sleep(1)

if health != 'OK':
    raise RuntimeError('Selkies did not start. Log tail:\n' +
                       (BENCH / 'selkies.log').read_text(errors='replace')[-5000:])

with urllib.request.urlopen('http://127.0.0.1:8080/api/status', timeout=5) as response:
    status = json.load(response)

time.sleep(2)
serverlog = (BENCH / 'selkies.log').read_text(errors='replace')

# =========================================================================
# EMPIRICAL STOREFRONT GUI LAUNCH & WINDOW VERIFICATION PROBE
# =========================================================================
print('▶ Starting empirical GUI launch tests for storefronts...', flush=True)
auth_path_file = BENCH / 'xauthority-path'
probe_env = dict(os.environ, DISPLAY=':0')
if auth_path_file.exists():
    probe_env['XAUTHORITY'] = auth_path_file.read_text().strip()

def get_wm_windows():
    try:
        res = subprocess.run(['wmctrl', '-l'], capture_output=True, text=True, timeout=5,
                             env=probe_env)
        return res.stdout.splitlines() if res.returncode == 0 else []
    except Exception:
        return []

def capture_screenshot(out_path):
    try:
        result = subprocess.run(['ffmpeg', '-f', 'x11grab', '-video_size', '1280x720', '-i', ':0',
                        '-vframes', '1', '-y', str(out_path)],
                       capture_output=True, timeout=10, env=probe_env)
        if result.returncode != 0:
            print('Screenshot failed:', result.stderr.decode(errors='replace')[-350:], flush=True)
        return out_path.exists() and out_path.stat().st_size > 1000
    except Exception:
        return False

smoke_tests = [
    {
        'id': 'minigalaxy',
        'title': 'Minigalaxy GOG Client',
        'script': str(BENCH / 'launch_minigalaxy.sh'),
        'match': 'minigalaxy',
        'log': '/home/gamer/logs/minigalaxy.log'
    },
    {
        'id': 'heroic',
        'title': 'Heroic Games Launcher',
        'script': str(BENCH / 'launch_heroic.sh'),
        'match': 'heroic',
        'log': '/home/gamer/logs/heroic.log'
    },
    {
        'id': 'lutris',
        'title': 'Lutris Gaming Hub',
        'script': str(BENCH / 'launch_lutris.sh'),
        'match': 'lutris',
        'log': '/home/gamer/logs/lutris.log'
    },
    {
        'id': 'steam',
        'title': 'Steam Native Client',
        'script': str(BENCH / 'launch_steam.sh'),
        'match': 'steam',
        'log': '/home/gamer/logs/steam.log'
    },
    {
        'id': 'ea_app',
        'title': 'EA App Wine Runner',
        'script': str(BENCH / 'launch_ea.sh'),
        'match': 'ea',
        'log': '/home/gamer/logs/ea.log'
    }
]

smoke_results = {}
for test in smoke_tests:
    tid = test['id']
    script = test['script']
    match_str = test['match'].lower()
    log_file = Path(test['log'])
    shot_file = SCREENSHOTS / f'{tid}_smoke.png'
    
    print(f'  Testing storefront GUI: {test["title"]}...', flush=True)
    if not os.path.exists(script):
        smoke_results[tid] = {'tested': False, 'error': 'Launch script missing'}
        continue
        
    proc = subprocess.Popen(['runuser', '-u', 'gamer', '--', 'env', '-u', 'DBUS_SESSION_BUS_ADDRESS',
                             'PULSE_SERVER=' + (BENCH / 'pulse-server-path').read_text().strip(),
                             'dbus-run-session', '--', '/bin/bash', script], start_new_session=True)
    window_found = False
    matching_titles = []
    
    # Steam cold setup often takes longer than storefront startup.
    max_polls = 60 if tid == 'steam' else 20 if tid == 'heroic' else 12
    for _ in range(max_polls):
        time.sleep(1.5)
        wins = get_wm_windows()
        for w in wins:
            if match_str in w.lower():
                window_found = True
                matching_titles.append(w.strip())
        if window_found:
            break
            
    # Capture screen regardless of window detection to document actual virtual display state
    all_windows = get_wm_windows()
    error_shot = None
    if tid == 'steam':
        error_windows = [line for line in all_windows if 'error' in line.lower()]
        if error_windows:
            subprocess.run(['wmctrl', '-ia', error_windows[-1].split()[0]],
                           capture_output=True, timeout=5, env=probe_env)
            time.sleep(1)
            error_path = SCREENSHOTS / 'steam_error.png'
            if capture_screenshot(error_path):
                error_shot = str(error_path)
    shot_captured = capture_screenshot(shot_file)
    
    # Read process log tail
    log_tail = []
    if log_file.exists():
        lines = [l.strip() for l in log_file.read_text(encoding='utf-8', errors='replace').splitlines() if l.strip()]
        log_tail = lines[-6:]
        
    smoke_results[tid] = {
        'tested': True,
        'pid': proc.pid,
        'window_mapped': window_found,
        'detected_windows': matching_titles,
        'all_windows': all_windows,
        'error_screenshot': error_shot,
        'screenshot_saved': shot_captured,
        'screenshot_path': str(shot_file) if shot_captured else None,
        'process_poll': proc.poll(),
        'log_tail': log_tail
    }
    
    # Terminate smoke test process so desktop stays clean
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        time.sleep(1)
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGKILL)
    except Exception:
        pass

subprocess.run(['wmctrl', '-a', 'Game Lab'], capture_output=True, timeout=5,
               env=probe_env)
time.sleep(1)

# Compile comprehensive health and verification audit
def check_bin(name):
    path = shutil.which(name) or (f'/usr/games/{name}' if os.path.exists(f'/usr/games/{name}') else None)
    return {'installed': path is not None, 'path': path}

def epic_installed_apps():
    installed = Path('/home/gamer/.config/heroic/legendaryConfig/legendary/installed.json')
    try:
        data = json.loads(installed.read_text())
        return sorted(data) if isinstance(data, dict) else []
    except (OSError, ValueError):
        return []

report = {
    'health': health,
    'status': status,
    'nvenc_detected': 'encodes H264, H265 on nvenc' in serverlog,
    'gamepad_sockets_ready': '4 virtual gamepads ready' in serverlog,
    'server_pid': server.pid,
    'launcher_processes': subprocess.run(['pgrep', '-u', 'gamer', '-f', '/tmp/kaggle-game-dev-bench/dev_launcher.py'],
                                         capture_output=True, text=True).stdout.splitlines(),
    'launcher_screenshot': str(SCREENSHOTS / 'game_lab_home.png') if capture_screenshot(SCREENSHOTS / 'game_lab_home.png') else None,
    'desktop_processes': [line for line in subprocess.run(['ps', '-eo', 'user,pid,ppid,args'],
                                      capture_output=True, text=True).stdout.splitlines()
                          if any(name in line for name in ('Xvfb', 'openbox', 'tint2', 'dev_launcher', 'selkies-session'))][:20],
    'x11_probe': subprocess.run(['xdpyinfo', '-display', ':0'], capture_output=True,
                                text=True, timeout=10, env=probe_env).stderr[-800:],
    'xauthority_path': probe_env.get('XAUTHORITY'),
    'gamer_audio_probe': subprocess.run(['runuser', '-u', 'gamer', '--', 'env',
                                         'PULSE_SERVER=' + (BENCH / 'pulse-server-path').read_text().strip(),
                                         'pactl', 'info'], capture_output=True,
                                         text=True, timeout=10).returncode,
    'selkies_log_tail': (BENCH / 'selkies.log').read_text(errors='replace')[-3000:],
    'desktop_components': {
        'openbox': check_bin('openbox'),
        'tint2': check_bin('tint2'),
        'xterm': check_bin('xterm'),
        'pulseaudio': check_bin('pulseaudio')
    },
    'installed_executables': {
        'steam': check_bin('steam'),
        'heroic': check_bin('heroic'),
        'legendary': check_bin('legendary'),
        'minigalaxy': check_bin('minigalaxy'),
        'lutris': check_bin('lutris'),
        'wine': check_bin('wine'),
        'wine_ge': {'installed': any(os.path.exists(p) for p in ['/home/gamer/.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine', *Path('/home/gamer/.config/heroic/tools/wine').glob('*/bin/wine')]), 'path': next((str(p) for p in ['/home/gamer/.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine', *Path('/home/gamer/.config/heroic/tools/wine').glob('*/bin/wine')] if os.path.exists(str(p))), None)}
    },
    'empirical_gui_smoke_tests': smoke_results,
    'epic_library_probe': {
        'heroic_legendary_config': '/home/gamer/.config/heroic/legendaryConfig/legendary',
        'heroic_installed_apps': epic_installed_apps(),
        'standalone_legendary_config': '/home/gamer/.config/legendary',
        'interactive_account_test_required': True
    },
    'compatibility_matrix': {
        'native_linux_2d_and_cpu_games': 'Fully supported (Mesa llvmpipe software rasterizer)',
        'native_linux_opengl_games': 'Supported via software rasterization (no hardware NVIDIA GLX scanout)',
        'native_vulkan_and_dxvk_games': 'UNAVAILABLE (datacenter Tesla T4 lacks container display/render nodes and ICD)',
        'windows_games_under_wine_proton': 'Supported ONLY with WineD3D (OpenGL translation); DXVK fails due to missing Vulkan',
        'epic_games_support': 'Heroic and Legendary installed; individual game launch needs authenticated interactive testing',
        'controller_support': '4 Xbox 360 userspace virtual gamepads via interposer',
        'audio_support': 'PulseAudio stereo monitor -> Selkies Opus 128 kbps',
        'video_support': 'Tesla T4 NVENC hardware H.264 encode at 60 FPS'
    }
}

try:
    listing = subprocess.run([sys.executable, '-m', 'jupyter', 'server', 'list', '--json'],
                             capture_output=True, text=True, timeout=15).stdout
    for line in listing.splitlines():
        if not line.lstrip().startswith('{'):
            continue
        item = json.loads(line)
        base = item.get('url', '').rstrip('/') + '/'
        token = item.get('token', '')
        if not base.startswith(('http://127.0.0.1:', 'http://localhost:')):
            continue
        candidate = base + 'proxy/8080/api/health'
        if token:
            candidate += '?token=' + token
        with urllib.request.urlopen(candidate, timeout=5) as response:
            report['jupyter_proxy_local'] = response.status == 200 and response.read() == b'OK'
        break
except Exception as exc:
    report['jupyter_proxy_local'] = False
    report['jupyter_proxy_error'] = type(exc).__name__

# Persist reports into output directory
Path('/kaggle/working/game_dev_bench_health.json').write_text(json.dumps(report, indent=2))
print('================== HEALTH & DIAGNOSTIC REPORT ==================')
print(json.dumps(report, indent=2))
print('================================================================')
print('Linux Gaming & Game-Dev Bench is running in the active session.')


In [ ]:
from IPython.display import display, HTML
display(HTML(r"""
<div style="font:16px system-ui, -apple-system, sans-serif;padding:24px;border-radius:16px;background:#0d1729;color:#f4f8ff;max-width:760px;box-shadow:0 8px 24px rgba(0,0,0,0.5)">
  <h2 style="margin-top:0;color:#4dcdf2;display:flex;align-items:center;gap:10px">
    <span>🎮</span> Linux Gaming & Game-Dev Environment
  </h2>
  <p style="color:#b0c4de;line-height:1.5">
    Temporary storefront and game-build test bench with <b>Steam</b>, <b>Heroic</b>, <b>Legendary</b>, <b>Minigalaxy</b>, <b>Lutris</b>, and an experimental <b>EA App</b> installer.
  </p>

  <div style="background:#132238;padding:16px;border-radius:12px;margin:16px 0;border-left:4px solid #4dcdf2">
    <b>Connection Steps:</b>
    <ol style="margin:8px 0 0 18px;padding:0;color:#c8daf0;line-height:1.6">
      <li>In Kaggle, open <b>Run → Kaggle Jupyter Server</b> and copy the <b>VSCode Compatible URL</b>.</li>
      <li>Paste it into the field below and click <b>Connect to Stream</b>.</li>
      <li>In the Selkies client, click the sound icon to unmute Opus stereo audio.</li>
      <li>Navigate the storefront launcher with your keyboard, mouse, or connected gamepad!</li>
    </ol>
  </div>

  <div style="margin:16px 0">
    <input id="kaggle-base" type="text" style="width:96%;padding:12px;border-radius:8px;border:1px solid #233b5d;background:#080e1a;color:#fff;font-size:14px" placeholder="Paste VSCode Compatible URL (https://...)">
    <button id="kaggle-open" style="margin-top:12px;padding:12px 24px;border-radius:8px;border:none;background:#4dcdf2;color:#081325;font-weight:bold;font-size:15px;cursor:pointer">Connect to Stream</button>
  </div>
  <p id="kaggle-result" style="font-size:13px;color:#9bdcf5;margin-top:6px"></p>

  <div style="border-top:1px solid #1a2a44;padding-top:14px;margin-top:18px;font-size:13px;color:#8ba2c4">
    <b>Empirical Hardware & Security Reality:</b>
    <ul style="margin:6px 0 0 16px;padding:0;line-height:1.5">
      <li><b>User state:</b> Storefronts run as <code>gamer</code>; their temporary credentials and caches are under <code>/home/gamer</code>.</li>
      <li><b>Storefront EULA:</b> Valve's Subscriber Agreement is presented to the user directly in the Steam GUI on first launch.</li>
      <li><b>Rendering Limits:</b> T4 NVENC hardware H.264 stream at 60 FPS verified. Native 3D uses software Mesa llvmpipe; datacenter T4 lacks container display nodes and Vulkan ICD.</li>
      <li><b>Custom Builds:</b> Upload test Linux builds into <code style="color:#4dcdf2">/tmp/game-builds</code> via Selkies Files panel.</li>
    </ul>
  </div>
</div>
<script>
document.getElementById('kaggle-open').onclick = function () {
  try {
    const raw = document.getElementById('kaggle-base').value.trim();
    if (!raw) throw new Error('Please paste your URL.');
    const u = new URL(raw);
    let p = u.pathname.replace(/\/(tree|lab|notebooks)(\/.*)?$/, '/');
    if (!p.endsWith('/')) p += '/';
    u.pathname = p + 'proxy/8080/';
    const target = u.toString();
    document.getElementById('kaggle-result').textContent = 'Opening authenticated Jupyter proxy route...';
    window.open(target, '_blank', 'noopener,noreferrer');
  } catch (e) {
    document.getElementById('kaggle-result').textContent = 'Please enter a valid https:// Kaggle Jupyter Server URL.';
  }
};
</script>
"""))
